# 📊 Project FORESIGHT

## AI-Powered Demand & Inventory Intelligence Platform

### Notebook 03: Feature Engineering

---

## 📌 Project Information

| Item | Details |
|---|---|
| **Author** | Mohit Kumar |
| **Internship** | Zidio Development |
| **Project Type** | End-to-End Data Analytics & Machine Learning Project |
| **Notebook Version** | 1.0 |
| **Status** | Completed |
| **Last Updated** | August 2026 |

<style>
@media print {
    .jp-OutputArea-output.jp-RenderedHTMLCommon {
        width: 100% !important;
        max-width: 100% !important;
        overflow-x: hidden !important;
    }

    .jp-OutputArea-output.jp-RenderedHTMLCommon table.dataframe {
        width: 100% !important;
        max-width: 100% !important;
        table-layout: fixed !important;
        font-size: 6pt !important;
    }

    .jp-OutputArea-output.jp-RenderedHTMLCommon table.dataframe th,
    .jp-OutputArea-output.jp-RenderedHTMLCommon table.dataframe td {
        padding: 2px 3px !important;
        white-space: normal !important;
        overflow-wrap: anywhere !important;
        word-break: break-word !important;
    }
}
</style>

---
# 🎯 1. Business Context and Notebook Objective

Demand forecasting requires more than clean historical data. Forecasting models need structured predictors that represent seasonality, recent demand behaviour, longer-term demand patterns, pricing, promotions and other information available before the forecast period.

Notebook 03 transforms the validated daily analytical base produced by Notebook 02 into a weekly, leakage-safe modelling dataset for the approved 300 SKU-store series.

The objectives of this notebook are to:

- create one complete weekly observation for each SKU-store series;
- define weekly units sold as the forecasting target;
- engineer historical demand, calendar, price, promotion and demand-behaviour features;
- identify weekly observations with sufficient preceding history for model eligibility;
- validate feature calculations and prevent future-information leakage;
- export the final dataset and feature metadata for forecasting.

This notebook performs feature engineering only. Forecasting models, validation periods, performance metrics and inventory recommendations will be developed in subsequent notebooks.

----
# 🧭 2. Notebook Workflow and Feature Engineering Framework

Notebook 03 follows a focused feature-engineering workflow designed to transform the validated daily analytical base produced by Notebook 02 into a weekly, leakage-safe modelling dataset for demand forecasting.

## 2.1 Notebook Workflow

| Section | Focus |
|---:|---|
| 3 | Notebook Setup and Input Data |
| 4 | Weekly Modelling Base |
| 5 | Forecasting Feature Engineering |
| 6 | Final Modelling Dataset |
| 7 | Feature Validation and Leakage Audit |
| 8 | Export and Validate Modelling Outputs |
| 9 | Conclusion and Notebook 04 Handoff |

## 2.2 Feature Engineering Framework

The modelling dataset will be constructed around the following components:

| Component | Purpose |
|---|---|
| **Weekly Target** | Define demand as weekly units sold for each SKU-store series |
| **Calendar Features** | Represent time progression and recurring seasonal patterns using forecast-time-available calendar information |
| **Historical Demand Features** | Represent prior demand behaviour through lagged and rolling statistics across short-, medium- and annual horizons |
| **Commercial and Calendar Drivers** | Represent historical price and promotion conditions together with scheduled-event and applicable SNAP information known before the target week |
| **Demand-Behaviour Features** | Describe prior-demand recency, activity and intermittency at each target week |
| **Hierarchy Features** | Preserve item, category, department, store and state context for later modelling |
| **History Eligibility** | Identify series-week observations with sufficient preceding history for model use |

<div style="page-break-before: always;"></div>

## 2.3 Feature Engineering Principles

- Use only the validated 300 SKU-store development scope.
- Build one complete weekly observation for each SKU-store series.
- Use weekly units sold as the forecasting target.
- Derive demand and commercial features only from information available before the target week.
- Use directly only those calendar variables that are known in advance.
- Treat missing values created by lags and rolling windows as structural.
- Retain categorical hierarchy fields for later encoding within the modelling pipeline.
- Keep simulated inventory variables outside the core demand-forecasting dataset.
- Create only features with a clear forecasting purpose.

> **Scope boundary:** This notebook prepares and validates forecasting features only. Evaluation periods, forecasting models, performance metrics and inventory decisions belong to subsequent notebooks.

---
# ⚙️ 3. Notebook Setup and Input Data

This section establishes the reproducible notebook environment, defines the feature-engineering configuration and loads the validated daily analytical base produced by Notebook 02. Formal input-contract and structural-readiness checks will verify that the dataset is suitable for weekly feature engineering.

## 3.1 Configure the Notebook

This subsection establishes the required Python environment, project paths, modelling input contract and central feature-engineering parameters. An input-availability check will confirm that the validated Notebook 02 handoff is accessible before it is loaded.

### 3.1.1 Library Imports and Display Settings

The required libraries will support data manipulation, numerical feature construction, project-path management and metadata export. Notebook-wide display settings will also be configured to keep validation outputs readable.

In [1]:
# Standard libraries
import json
from pathlib import Path

# Data manipulation
import numpy as np
import pandas as pd

# Notebook display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

print("✅ Required libraries imported and display settings configured.")

✅ Required libraries imported and display settings configured.


### 📝 Library Summary

The required Python libraries were successfully imported and configured for Notebook 03. Each library supports a specific part of the feature-engineering workflow.

| Library | Purpose |
|---|---|
| **pandas** | Data loading, weekly aggregation, feature construction and validation |
| **NumPy** | Numerical calculations, cyclical transformations and missing-value handling |
| **pathlib** | Project-directory and file-path management |
| **json** | Export of approved feature lists and supporting metadata |

The notebook environment is now prepared for input validation, weekly dataset construction and leakage-safe feature engineering.

### 3.1.2 Project Paths and Input Contract

Notebook 03 will use the validated daily analytical base exported by Notebook 02 as its single modelling input. Centralised project paths will support reproducible input loading and later output generation.

The modelling-output directory is defined here but will not be created until the completed outputs are exported in Section 8.

In [2]:
# Identify the project root
current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    PROJECT_ROOT = current_dir.parent
else:
    PROJECT_ROOT = current_dir

# Define project directories
INTERIM_DATA_DIR = PROJECT_ROOT / "data" / "interim"
MODELING_DATA_DIR = PROJECT_ROOT / "data" / "modeling"

# Define the validated Notebook 02 handoff
ANALYTICS_BASE_FILE = INTERIM_DATA_DIR / "analytics_base_daily.parquet"

print(f"Project root              : {PROJECT_ROOT}")
print(f"Interim data directory    : {INTERIM_DATA_DIR}")
print(f"Modeling output directory : {MODELING_DATA_DIR}")
print(f"Analytical base           : {ANALYTICS_BASE_FILE}")

Project root              : C:\Users\hp\Project FORESIGHT
Interim data directory    : C:\Users\hp\Project FORESIGHT\data\interim
Modeling output directory : C:\Users\hp\Project FORESIGHT\data\modeling
Analytical base           : C:\Users\hp\Project FORESIGHT\data\interim\analytics_base_daily.parquet


### 3.1.3 Feature-Engineering Configuration

The approved development scope, minimum history requirement and historical-demand windows will be defined centrally. These parameters will be reused consistently during feature construction and validation.

The feature groups and governing principles are fixed, but the final feature list will be confirmed only after the available history and individual feature behaviour have been examined.

In [3]:
# Development-scope configuration
EXPECTED_SERIES_COUNT = 300
MIN_HISTORY_WEEKS = 52

# Historical-demand feature configuration
DEMAND_LAGS = [1, 2, 4, 8, 13, 26, 52]
ROLLING_MEAN_WINDOWS = [4, 8, 13, 26, 52]
ROLLING_STD_WINDOWS = [4, 13, 52]

# Present the central configuration
configuration_summary = pd.DataFrame(
    {
        "parameter": [
            "Expected SKU-store series",
            "Minimum history requirement",
            "Demand lags",
            "Rolling-mean windows",
            "Rolling-standard-deviation windows",
        ],
        "value": [
            EXPECTED_SERIES_COUNT,
            f"{MIN_HISTORY_WEEKS} weeks",
            ", ".join(map(str, DEMAND_LAGS)),
            ", ".join(map(str, ROLLING_MEAN_WINDOWS)),
            ", ".join(map(str, ROLLING_STD_WINDOWS)),
        ],
    }
)

configuration_summary

,parameter,value
0,Expected SKU-store series,300
1,Minimum history requirement,52 weeks
2,Demand lags,"1, 2, 4, 8, 13, 26, 52"
3,Rolling-mean windows,"4, 8, 13, 26, 52"
4,Rolling-standard-deviation windows,"4, 13, 52"


### 3.1.4 Input Availability Check

The required Notebook 02 handoff will be checked before it is loaded. This prevents feature engineering from beginning with a missing or incorrectly located input file.

In [4]:
# Check the required Notebook 02 handoff without loading it
input_check = pd.DataFrame(
    [
        {
            "dataset": "Daily analytical base",
            "relative_path": str(
                ANALYTICS_BASE_FILE.relative_to(PROJECT_ROOT)
            ),
            "exists": ANALYTICS_BASE_FILE.exists(),
            "file_size_mb": (
                ANALYTICS_BASE_FILE.stat().st_size / (1024**2)
                if ANALYTICS_BASE_FILE.exists()
                else np.nan
            ),
        }
    ]
)

input_check

,dataset,relative_path,exists,file_size_mb
0,Daily analytical base,data\interim\analytics_base_daily.parquet,True,4.60


In [5]:
# Enforce availability of the required modelling input
assert ANALYTICS_BASE_FILE.exists(), (
    "Required Notebook 02 handoff was not found: "
    f"{ANALYTICS_BASE_FILE}"
)

print("✅ Required Notebook 02 handoff is available.")

✅ Required Notebook 02 handoff is available.


### ✅ Configuration Summary

Notebook 03 is now configured for reproducible feature engineering:

- the required libraries and display settings have been established;
- the project root and modelling paths have been defined;
- the validated daily analytical base has been established as the single modelling input;
- the development scope and historical-demand parameters have been centralised;
- the required Notebook 02 handoff has been located successfully.

The notebook environment is ready to load and verify the daily analytical base.

---
## 3.2 Load and Verify the Daily Analytical Base

The validated daily analytical base produced by Notebook 02 will now be loaded as the single input to the feature-engineering workflow.

Before any transformation is performed, the dataset will be checked against its documented handoff contract. The validation will cover:

- dataset dimensions and date coverage;
- required modelling fields;
- data types and missingness;
- uniqueness and completeness of the daily analytical key;
- preservation of the approved 300-series development scope;
- balanced daily panel coverage;
- chronological ordering and target validity.

These checks create a controlled boundary between the validated analytical layer and the downstream modelling layer.

### 3.2.1 Load the Validated Analytical Base

The Parquet handoff will be loaded without modifying its values or structure. A separate working dataset will be created later when weekly transformations begin.

In [6]:
# Load the validated Notebook 02 analytical base
analytics_daily = pd.read_parquet(ANALYTICS_BASE_FILE)

print("✅ Daily analytical base loaded successfully.")
print(f"Rows    : {analytics_daily.shape[0]:,}")
print(f"Columns : {analytics_daily.shape[1]:,}")

✅ Daily analytical base loaded successfully.
Rows    : 573,900
Columns : 56


### 3.2.2 Input Snapshot

A concise snapshot will establish the loaded dataset's dimensions, memory footprint, series coverage and date coverage before detailed validation begins.

In [7]:
# Create a high-level snapshot of the loaded input
input_snapshot = pd.DataFrame(
    {
        "metric": [
            "Rows",
            "Columns",
            "In-memory size (MB)",
            "Unique SKU-store series",
            "Unique dates",
            "Minimum date",
            "Maximum date",
        ],
        "value": [
            f"{analytics_daily.shape[0]:,}",
            f"{analytics_daily.shape[1]:,}",
            f"{analytics_daily.memory_usage(deep=True).sum() / (1024**2):,.2f}",
            (
                f"{analytics_daily['sku_id'].nunique():,}"
                if "sku_id" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                f"{analytics_daily['date'].nunique():,}"
                if "date" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                str(analytics_daily["date"].min())
                if "date" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                str(analytics_daily["date"].max())
                if "date" in analytics_daily.columns
                else "Unavailable"
            ),
        ],
    }
)

input_snapshot

,metric,value
0,Rows,"573,900"
1,Columns,56
2,In-memory size (MB),313.18
3,Unique SKU-store series,300
4,Unique dates,"1,913"
5,Minimum date,2011-01-29 00:00:00
6,Maximum date,2016-04-24 00:00:00


### 3.2.3 Schema and Missingness Inspection

The complete input schema will be inspected before modelling fields are selected. This check records each column's data type, non-null coverage and missing-value count without altering the dataset.

In [8]:
# Build a column-level schema and completeness profile
schema_profile = pd.DataFrame(
    {
        "column": analytics_daily.columns,
        "dtype": analytics_daily.dtypes.astype(str).values,
        "non_null_rows": analytics_daily.notna().sum().values,
        "missing_rows": analytics_daily.isna().sum().values,
        "missing_pct": (
            analytics_daily.isna().mean().mul(100).values
        ),
    }
)

schema_profile

,column,dtype,non_null_rows,missing_rows,missing_pct
0,date,datetime64[us],573900,0,0.00
1,sku_id,str,573900,0,0.00
2,units_sold,int64,573900,0,0.00
3,revenue,float64,573900,0,0.00
4,unit_price,float64,573900,0,0.00
5,promo_flag,int64,573900,0,0.00
6,d,str,573900,0,0.00
7,wm_yr_wk,int64,573900,0,0.00
8,year,int64,573900,0,0.00
9,quarter,int64,573900,0,0.00


In [9]:
# Summarize the distribution of input data types
dtype_summary = (
    schema_profile
    .groupby("dtype", as_index=False)
    .agg(column_count=("column", "count"))
    .sort_values("column_count", ascending=False)
    .reset_index(drop=True)
)

dtype_summary

,dtype,column_count
0,int64,28
1,str,19
2,float64,8
3,datetime64[us],1


<div style="break-before: page; page-break-before: always;"></div>

### 3.2.4 Core Input Contract

The analytical base is expected to preserve the verified Notebook 02 handoff and provide the key, chronology and target fields required for preliminary structural validation.

This preliminary contract is intentionally narrower than the complete weekly source-field contract enforced in Section 3.2.7.

The contract distinguishes between:

- **snapshot expectations**, which reconcile the loaded file with the documented Notebook 02 export;
- **core structural requirements**, which must hold before detailed panel validation can proceed.

In [10]:
# Document the verified Notebook 02 snapshot
EXPECTED_INPUT_ROWS = 573_900
EXPECTED_INPUT_COLUMNS = 56
EXPECTED_DAILY_DATES = 1_913
EXPECTED_START_DATE = pd.Timestamp("2011-01-29")
EXPECTED_END_DATE = pd.Timestamp("2016-04-24")

# Define the minimum fields required for weekly modelling
REQUIRED_INPUT_COLUMNS = [
    "sku_id",
    "date",
    "units_sold",
]

missing_required_columns = sorted(
    set(REQUIRED_INPUT_COLUMNS) - set(analytics_daily.columns)
)

input_contract = pd.DataFrame(
    {
        "contract_item": [
            "Expected input rows",
            "Expected input columns",
            "Expected daily dates",
            "Expected date range",
            "Expected SKU-store series",
            "Required modelling fields",
        ],
        "expected_value": [
            f"{EXPECTED_INPUT_ROWS:,}",
            EXPECTED_INPUT_COLUMNS,
            f"{EXPECTED_DAILY_DATES:,}",
            f"{EXPECTED_START_DATE.date()} to {EXPECTED_END_DATE.date()}",
            EXPECTED_SERIES_COUNT,
            ", ".join(REQUIRED_INPUT_COLUMNS),
        ],
        "observed_value": [
            f"{analytics_daily.shape[0]:,}",
            analytics_daily.shape[1],
            (
                f"{analytics_daily['date'].nunique():,}"
                if "date" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                f"{analytics_daily['date'].min()} to "
                f"{analytics_daily['date'].max()}"
                if "date" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                analytics_daily["sku_id"].nunique()
                if "sku_id" in analytics_daily.columns
                else "Unavailable"
            ),
            (
                "All available"
                if not missing_required_columns
                else ", ".join(missing_required_columns)
            ),
        ],
    }
)

input_contract

,contract_item,expected_value,observed_value
0,Expected input rows,"573,900","573,900"
1,Expected input columns,56,56
2,Expected daily dates,"1,913","1,913"
3,Expected date range,2011-01-29 to 2016-04-24,2011-01-29 00:00:00 to 2016-04-24 00:00:00
4,Expected SKU-store series,300,300
5,Required modelling fields,"sku_id, date, units_sold",All available


### 3.2.5 Structural and Panel Validation

The following checks will verify that the input remains a complete and uniquely keyed daily panel. These conditions are required before observations can be aggregated into weekly modelling records.

In [11]:
# Stop immediately if a core modelling field is unavailable
assert not missing_required_columns, (
    "Required input columns are missing: "
    f"{missing_required_columns}"
)

# Standardize the date field only when necessary
if not pd.api.types.is_datetime64_any_dtype(analytics_daily["date"]):
    analytics_daily["date"] = pd.to_datetime(
        analytics_daily["date"],
        errors="raise",
    )

print("✅ Core modelling fields are available.")
print(f"Date dtype: {analytics_daily['date'].dtype}")

✅ Core modelling fields are available.
Date dtype: datetime64[us]


In [12]:
# Calculate the principal structural-validation measures
duplicate_key_count = analytics_daily.duplicated(
    subset=["sku_id", "date"]
).sum()

missing_key_count = analytics_daily[
    ["sku_id", "date"]
].isna().sum().sum()

series_count = analytics_daily["sku_id"].nunique()
date_count = analytics_daily["date"].nunique()

records_per_series = analytics_daily.groupby(
    "sku_id",
    observed=True,
).size()

series_per_date = analytics_daily.groupby(
    "date",
    observed=True,
)["sku_id"].nunique()

negative_target_count = (
    analytics_daily["units_sold"].lt(0).sum()
)

structural_metrics = pd.DataFrame(
    {
        "metric": [
            "Duplicate sku_id-date keys",
            "Missing analytical-key values",
            "Unique SKU-store series",
            "Unique dates",
            "Minimum records per series",
            "Maximum records per series",
            "Minimum series per date",
            "Maximum series per date",
            "Missing target values",
            "Negative target values",
        ],
        "observed_value": [
            duplicate_key_count,
            missing_key_count,
            series_count,
            date_count,
            records_per_series.min(),
            records_per_series.max(),
            series_per_date.min(),
            series_per_date.max(),
            analytics_daily["units_sold"].isna().sum(),
            negative_target_count,
        ],
    }
)

structural_metrics

,metric,observed_value
0,Duplicate sku_id-date keys,0
1,Missing analytical-key values,0
2,Unique SKU-store series,300
3,Unique dates,1913
4,Minimum records per series,1913
5,Maximum records per series,1913
6,Minimum series per date,300
7,Maximum series per date,300
8,Missing target values,0
9,Negative target values,0


### 3.2.6 Chronology Inspection

A chronological sample will confirm that the analytical key, target and selected source attributes align correctly at the daily grain. The complete dataset will remain unchanged; sorting will be applied only to the displayed sample.

In [13]:
# Display a focused chronological sample using verified source fields
preview_columns = [
    "sku_id",
    "date",
    "units_sold",
    "unit_price",
    "promo_flag",
    "event_name_1",
    "event_flag",
    "snap_flag",
]

chronological_sample = (
    analytics_daily
    .sort_values(["sku_id", "date"])
    .loc[:, preview_columns]
    .head(10)
)

chronological_sample

,sku_id,date,units_sold,unit_price,promo_flag,event_name_1,event_flag,snap_flag
0,FOODS_1_001_CA_1,2011-01-29,3,2.00,0,NaN,0,0
1,FOODS_1_001_CA_1,2011-01-30,0,2.00,0,NaN,0,0
2,FOODS_1_001_CA_1,2011-01-31,0,2.00,0,NaN,0,0
3,FOODS_1_001_CA_1,2011-02-01,1,2.00,0,NaN,0,1
4,FOODS_1_001_CA_1,2011-02-02,4,2.00,0,NaN,0,1
5,FOODS_1_001_CA_1,2011-02-03,2,2.00,0,NaN,0,1
6,FOODS_1_001_CA_1,2011-02-04,0,2.00,0,NaN,0,1
7,FOODS_1_001_CA_1,2011-02-05,2,2.00,0,NaN,0,1
8,FOODS_1_001_CA_1,2011-02-06,0,2.00,0,SuperBowl,1,1
9,FOODS_1_001_CA_1,2011-02-07,0,2.00,0,NaN,0,1


### 3.2.7 Input Validation Register

The executed input checks will now be consolidated into a formal validation register. Each control compares the observed analytical base against either the documented Notebook 02 snapshot or a structural requirement of the weekly feature-engineering pipeline.

The required source-field contract includes only the variables needed to construct the weekly target, hierarchy, commercial, event and SNAP features. Inventory-derived fields and current-period revenue are not approved as forecasting predictors at this stage.

In [14]:
# Define the source fields required by the weekly feature-engineering pipeline
REQUIRED_WEEKLY_SOURCE_COLUMNS = [
    "sku_id",
    "date",
    "wm_yr_wk",
    "units_sold",
    "unit_price",
    "promo_flag",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "event_flag",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "snap_flag",
    "item_id",
    "category",
    "department",
    "store_id",
    "state_id",
]

missing_weekly_source_columns = sorted(
    set(REQUIRED_WEEKLY_SOURCE_COLUMNS) - set(analytics_daily.columns)
)

# Identify columns containing missing values
observed_missing_columns = sorted(
    schema_profile.loc[
        schema_profile["missing_rows"].gt(0),
        "column",
    ].tolist()
)

# Event descriptions are naturally absent on non-event dates
EXPECTED_SPARSE_INPUT_COLUMNS = {
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
}

# Check continuity of the daily calendar
ordered_dates = (
    analytics_daily["date"]
    .drop_duplicates()
    .sort_values()
    .reset_index(drop=True)
)

non_daily_gap_count = (
    ordered_dates
    .diff()
    .dropna()
    .ne(pd.Timedelta(days=1))
    .sum()
)

# Store observed date boundaries
observed_start_date = pd.Timestamp(
    analytics_daily["date"].min()
).normalize()

observed_end_date = pd.Timestamp(
    analytics_daily["date"].max()
).normalize()

# Consolidate the executed input controls
validation_records = [
    {
        "control": "Snapshot row count",
        "expected": f"{EXPECTED_INPUT_ROWS:,}",
        "observed": f"{analytics_daily.shape[0]:,}",
        "passed": analytics_daily.shape[0] == EXPECTED_INPUT_ROWS,
    },
    {
        "control": "Snapshot column count",
        "expected": str(EXPECTED_INPUT_COLUMNS),
        "observed": str(analytics_daily.shape[1]),
        "passed": analytics_daily.shape[1] == EXPECTED_INPUT_COLUMNS,
    },
    {
        "control": "Daily date count",
        "expected": f"{EXPECTED_DAILY_DATES:,}",
        "observed": f"{date_count:,}",
        "passed": date_count == EXPECTED_DAILY_DATES,
    },
    {
        "control": "Date range",
        "expected": (
            f"{EXPECTED_START_DATE.date()} to "
            f"{EXPECTED_END_DATE.date()}"
        ),
        "observed": (
            f"{observed_start_date.date()} to "
            f"{observed_end_date.date()}"
        ),
        "passed": (
            observed_start_date == EXPECTED_START_DATE
            and observed_end_date == EXPECTED_END_DATE
        ),
    },
    {
        "control": "Required weekly source fields",
        "expected": (
            f"All {len(REQUIRED_WEEKLY_SOURCE_COLUMNS)} fields available"
        ),
        "observed": (
            "All available"
            if not missing_weekly_source_columns
            else ", ".join(missing_weekly_source_columns)
        ),
        "passed": not missing_weekly_source_columns,
    },
    {
        "control": "Date data type",
        "expected": "Datetime-compatible",
        "observed": str(analytics_daily["date"].dtype),
        "passed": pd.api.types.is_datetime64_any_dtype(
            analytics_daily["date"]
        ),
    },
    {
        "control": "Unique sku_id-date key",
        "expected": "0 duplicates",
        "observed": f"{duplicate_key_count:,} duplicates",
        "passed": duplicate_key_count == 0,
    },
    {
        "control": "Analytical-key completeness",
        "expected": "0 missing values",
        "observed": f"{missing_key_count:,} missing values",
        "passed": missing_key_count == 0,
    },
    {
        "control": "SKU-store development scope",
        "expected": f"{EXPECTED_SERIES_COUNT:,} series",
        "observed": f"{series_count:,} series",
        "passed": series_count == EXPECTED_SERIES_COUNT,
    },
    {
        "control": "Records per series",
        "expected": f"{EXPECTED_DAILY_DATES:,} for every series",
        "observed": (
            f"{records_per_series.min():,} to "
            f"{records_per_series.max():,}"
        ),
        "passed": records_per_series.eq(
            EXPECTED_DAILY_DATES
        ).all(),
    },
    {
        "control": "Series per date",
        "expected": f"{EXPECTED_SERIES_COUNT:,} on every date",
        "observed": (
            f"{series_per_date.min():,} to "
            f"{series_per_date.max():,}"
        ),
        "passed": series_per_date.eq(
            EXPECTED_SERIES_COUNT
        ).all(),
    },
    {
        "control": "Complete panel row count",
        "expected": (
            f"{EXPECTED_SERIES_COUNT * EXPECTED_DAILY_DATES:,}"
        ),
        "observed": f"{analytics_daily.shape[0]:,}",
        "passed": (
            analytics_daily.shape[0]
            == EXPECTED_SERIES_COUNT * EXPECTED_DAILY_DATES
        ),
    },
    {
        "control": "Daily calendar continuity",
        "expected": "0 non-daily gaps",
        "observed": f"{non_daily_gap_count:,} non-daily gaps",
        "passed": non_daily_gap_count == 0,
    },
    {
        "control": "Target data type",
        "expected": "Numeric",
        "observed": str(analytics_daily["units_sold"].dtype),
        "passed": pd.api.types.is_numeric_dtype(
            analytics_daily["units_sold"]
        ),
    },
    {
        "control": "Target completeness",
        "expected": "0 missing values",
        "observed": (
            f"{analytics_daily['units_sold'].isna().sum():,} "
            "missing values"
        ),
        "passed": analytics_daily["units_sold"].notna().all(),
    },
    {
        "control": "Target non-negativity",
        "expected": "0 negative values",
        "observed": f"{negative_target_count:,} negative values",
        "passed": negative_target_count == 0,
    },
    {
        "control": "Input missingness pattern",
        "expected": "Limited to sparse event-description fields",
        "observed": (
            ", ".join(observed_missing_columns)
            if observed_missing_columns
            else "No missing columns"
        ),
        "passed": (
            set(observed_missing_columns)
            .issubset(EXPECTED_SPARSE_INPUT_COLUMNS)
            and analytics_daily["event_flag"].notna().all()
        ),
    },
]

input_validation_register = pd.DataFrame(validation_records)

input_validation_register["status"] = np.where(
    input_validation_register["passed"],
    "PASS",
    "FAIL",
)

input_validation_register = input_validation_register[
    ["control", "expected", "observed", "status"]
]

input_validation_register

,control,expected,observed,status
0,Snapshot row count,"573,900","573,900",PASS
1,Snapshot column count,56,56,PASS
2,Daily date count,"1,913","1,913",PASS
3,Date range,2011-01-29 to 2016-04-24,2011-01-29 to 2016-04-24,PASS
4,Required weekly source fields,All 20 fields available,All available,PASS
5,Date data type,Datetime-compatible,datetime64[us],PASS
6,Unique sku_id-date key,0 duplicates,0 duplicates,PASS
7,Analytical-key completeness,0 missing values,0 missing values,PASS
8,SKU-store development scope,300 series,300 series,PASS
9,Records per series,"1,913 for every series","1,913 to 1,913",PASS


In [15]:
# Enforce successful completion of every input control
failed_input_checks = input_validation_register.loc[
    input_validation_register["status"].eq("FAIL"),
    "control",
].tolist()

assert not failed_input_checks, (
    "Input validation failed for the following controls:\n- "
    + "\n- ".join(failed_input_checks)
)

print(
    "✅ Input validation passed: "
    f"{len(input_validation_register)} of "
    f"{len(input_validation_register)} controls passed."
)

✅ Input validation passed: 17 of 17 controls passed.


### ✅ Input Validation Result

The daily analytical base produced by Notebook 02 passed all input-contract and structural-readiness checks executed in Notebook 03:

- the dataset contains **573,900 daily observations and 56 source columns**;
- the approved development scope contains exactly **300 SKU-store series** across **1,913 consecutive dates** from **29 January 2011 to 24 April 2016**;
- every series contains 1,913 daily records and every date contains all 300 series;
- the `sku_id`–`date` analytical key is complete and unique;
- weekly feature-engineering source fields are available with valid date and target data types;
- `units_sold` contains no missing or negative values;
- missingness is limited to event-description fields, where null values correctly represent non-event dates and are supported by a complete binary event indicator.

The daily analytical base is therefore approved for weekly aggregation and leakage-controlled feature engineering.
**✅ Section 3 is complete.**

-------

<div style="page-break-before: always;"></div>

# 📅 4. Weekly Modelling Base

This section aggregates the validated daily analytical base into one record per SKU-store retail week using the existing `wm_yr_wk` identifier. Weekly units sold will become the forecasting target, while commercial and calendar variables will be summarized according to their meaning.

Only retail weeks containing all seven daily observations will be retained.

## 4.1 Construct the Weekly Modelling Base and Target

Daily observations will be aggregated by SKU-store series and retail week. Weekly units sold will be calculated as the forecasting target, relevant price, promotion, event and SNAP information will be summarized, and stable product-store attributes will be retained.
> **SNAP modelling rule:** The source `snap_flag` identifies SNAP activity in any tracked state. Because the modelling grain is SKU-store-week, the applicable SNAP indicator will instead be derived from each series's `state_id` and corresponding state-level SNAP field.

In [16]:
# Select only the daily fields required for weekly construction
daily_weekly_source = analytics_daily[
    [
        "sku_id",
        "date",
        "wm_yr_wk",
        "units_sold",
        "unit_price",
        "promo_flag",
        "event_type_1",
        "event_type_2",
        "event_flag",
        "snap_CA",
        "snap_TX",
        "snap_WI",
        "item_id",
        "category",
        "department",
        "store_id",
        "state_id",
    ]
].copy()

# Define the applicable state-level SNAP source
STATE_SNAP_COLUMNS = {
    "CA": "snap_CA",
    "TX": "snap_TX",
    "WI": "snap_WI",
}

unsupported_snap_states = sorted(
    set(daily_weekly_source["state_id"].dropna().unique())
    - set(STATE_SNAP_COLUMNS)
)

assert not unsupported_snap_states, (
    "Unsupported states found for SNAP mapping: "
    f"{unsupported_snap_states}"
)

daily_weekly_source["snap_applicable"] = np.select(
    [
        daily_weekly_source["state_id"].eq(state_id)
        for state_id in STATE_SNAP_COLUMNS
    ],
    [
        daily_weekly_source[snap_column]
        for snap_column in STATE_SNAP_COLUMNS.values()
    ],
    default=-1,
).astype("int8")

assert daily_weekly_source["snap_applicable"].isin([0, 1]).all(), (
    "Invalid state-applicable SNAP values were created."
)

# Create compact scheduled-event type indicators
EVENT_TYPES = [
    "Cultural",
    "National",
    "Religious",
    "Sporting",
]

for event_type in EVENT_TYPES:
    feature_name = f"{event_type.lower()}_event_flag"

    daily_weekly_source[feature_name] = (
        daily_weekly_source[
            ["event_type_1", "event_type_2"]
        ]
        .eq(event_type)
        .any(axis=1)
        .astype("int8")
    )

# Aggregate the daily input to the SKU-store retail-week grain
weekly_base_all = (
    daily_weekly_source
    .groupby(
        ["sku_id", "wm_yr_wk"],
        as_index=False,
        observed=True,
    )
    .agg(
        week_start_date=("date", "min"),
        week_end_date=("date", "max"),
        days_observed=("date", "nunique"),
        weekly_units_sold=("units_sold", "sum"),
        unit_price=("unit_price", "first"),
        promo_week_flag=("promo_flag", "max"),
        event_days=("event_flag", "sum"),
        event_week_flag=("event_flag", "max"),
        cultural_event_flag=("cultural_event_flag", "max"),
        national_event_flag=("national_event_flag", "max"),
        religious_event_flag=("religious_event_flag", "max"),
        sporting_event_flag=("sporting_event_flag", "max"),
        snap_active_days=("snap_applicable", "sum"),
        snap_week_flag=("snap_applicable", "max"),
        item_id=("item_id", "first"),
        category=("category", "first"),
        department=("department", "first"),
        store_id=("store_id", "first"),
        state_id=("state_id", "first"),
    )
)

# Retain only comparable seven-day retail weeks
complete_week_mask = weekly_base_all["days_observed"].eq(7)

partial_week_records = weekly_base_all.loc[
    ~complete_week_mask
].copy()

weekly_base = (
    weekly_base_all
    .loc[complete_week_mask]
    .sort_values(["sku_id", "week_start_date"])
    .reset_index(drop=True)
)

print("✅ Weekly modelling base constructed successfully.")

✅ Weekly modelling base constructed successfully.


In [17]:
# Summarize the weekly construction result
weekly_build_summary = pd.DataFrame(
    {
        "metric": [
            "Candidate SKU-store-week records",
            "Complete weekly records retained",
            "Partial weekly records excluded",
            "Complete retail weeks",
            "SKU-store series retained",
            "First complete week start",
            "Last complete week end",
        ],
        "value": [
            f"{len(weekly_base_all):,}",
            f"{len(weekly_base):,}",
            f"{len(partial_week_records):,}",
            f"{weekly_base['wm_yr_wk'].nunique():,}",
            f"{weekly_base['sku_id'].nunique():,}",
            str(weekly_base["week_start_date"].min().date()),
            str(weekly_base["week_end_date"].max().date()),
        ],
    }
)

weekly_build_summary

,metric,value
0,Candidate SKU-store-week records,"82,200"
1,Complete weekly records retained,"81,900"
2,Partial weekly records excluded,300
3,Complete retail weeks,273
4,SKU-store series retained,300
5,First complete week start,2011-01-29
6,Last complete week end,2016-04-22


In [18]:
# Display a focused sample of the constructed weekly records
weekly_preview = weekly_base[
    [
        "sku_id",
        "wm_yr_wk",
        "week_start_date",
        "week_end_date",
        "days_observed",
        "weekly_units_sold",
        "unit_price",
        "promo_week_flag",
        "event_days",
        "snap_active_days",
        "category",
        "store_id",
    ]
].head(10)

weekly_preview

,sku_id,wm_yr_wk,week_start_date,week_end_date,days_observed,weekly_units_sold,unit_price,promo_week_flag,event_days,snap_active_days,category,store_id
0,FOODS_1_001_CA_1,11101,2011-01-29,2011-02-04,7,10,2.00,0,0,4,FOODS,CA_1
1,FOODS_1_001_CA_1,11102,2011-02-05,2011-02-11,7,6,2.00,0,1,6,FOODS,CA_1
2,FOODS_1_001_CA_1,11103,2011-02-12,2011-02-18,7,10,2.00,0,1,0,FOODS,CA_1
3,FOODS_1_001_CA_1,11104,2011-02-19,2011-02-25,7,13,2.00,0,1,0,FOODS,CA_1
4,FOODS_1_001_CA_1,11105,2011-02-26,2011-03-04,7,15,2.00,0,0,4,FOODS,CA_1
5,FOODS_1_001_CA_1,11106,2011-03-05,2011-03-11,7,11,2.00,0,1,6,FOODS,CA_1
6,FOODS_1_001_CA_1,11107,2011-03-12,2011-03-18,7,7,2.00,0,2,0,FOODS,CA_1
7,FOODS_1_001_CA_1,11108,2011-03-19,2011-03-25,7,6,2.00,0,1,0,FOODS,CA_1
8,FOODS_1_001_CA_1,11109,2011-03-26,2011-04-01,7,5,2.00,0,0,1,FOODS,CA_1
9,FOODS_1_001_CA_1,11110,2011-04-02,2011-04-08,7,6,2.00,0,0,7,FOODS,CA_1


<div style="break-before: page; page-break-before: always;"></div>

## 4.2 Verify the Weekly Grain

Focused checks will confirm the unique `sku_id`–`wm_yr_wk` grain, complete seven-day weeks, preservation of all 300 series and reconciliation of weekly target values with the corresponding retained daily demand.

In [19]:
# Verify the weekly grain and reconcile the forecasting target
weekly_duplicate_key_count = weekly_base.duplicated(
    subset=["sku_id", "wm_yr_wk"]
).sum()

week_span_days = (
    weekly_base["week_end_date"]
    - weekly_base["week_start_date"]
).dt.days + 1

invalid_complete_week_count = (
    ~weekly_base["days_observed"].eq(7)
    | ~week_span_days.eq(7)
).sum()

weekly_series_count = weekly_base["sku_id"].nunique()
weekly_week_count = weekly_base["wm_yr_wk"].nunique()

weekly_records_per_series = weekly_base.groupby(
    "sku_id",
    observed=True,
).size()

weekly_series_per_week = weekly_base.groupby(
    "wm_yr_wk",
    observed=True,
)["sku_id"].nunique()

weekly_chronology = weekly_base.sort_values(
    ["sku_id", "week_start_date"]
)

weekly_gap_count = (
    weekly_chronology
    .groupby("sku_id", observed=True)["week_start_date"]
    .diff()
    .dropna()
    .ne(pd.Timedelta(days=7))
    .sum()
)

# Recalculate weekly demand directly from the retained daily records
retained_week_keys = weekly_base[
    ["sku_id", "wm_yr_wk"]
].drop_duplicates()

retained_daily = daily_weekly_source.merge(
    retained_week_keys,
    on=["sku_id", "wm_yr_wk"],
    how="inner",
    validate="many_to_one",
)

recalculated_weekly_target = (
    retained_daily
    .groupby(
        ["sku_id", "wm_yr_wk"],
        as_index=False,
        observed=True,
    )
    .agg(
        recalculated_units_sold=("units_sold", "sum")
    )
)

target_reconciliation = weekly_base[
    ["sku_id", "wm_yr_wk", "weekly_units_sold"]
].merge(
    recalculated_weekly_target,
    on=["sku_id", "wm_yr_wk"],
    how="left",
    validate="one_to_one",
)

weekly_target_mismatch_count = (
    ~np.isclose(
        target_reconciliation["weekly_units_sold"],
        target_reconciliation["recalculated_units_sold"],
    )
).sum()

weekly_validation_register = pd.DataFrame(
    {
        "control": [
            "Unique SKU-store-week key",
            "Complete seven-day weeks",
            "Approved series retained",
            "Records per series",
            "Series per retail week",
            "Continuous weekly chronology",
            "Weekly target reconciliation",
        ],
        "expected": [
            "0 duplicates",
            "0 invalid records",
            f"{EXPECTED_SERIES_COUNT:,} series",
            f"{weekly_week_count:,} for every series",
            f"{EXPECTED_SERIES_COUNT:,} for every week",
            "0 weekly gaps",
            "0 mismatches",
        ],
        "observed": [
            f"{weekly_duplicate_key_count:,} duplicates",
            f"{invalid_complete_week_count:,} invalid records",
            f"{weekly_series_count:,} series",
            (
                f"{weekly_records_per_series.min():,} to "
                f"{weekly_records_per_series.max():,}"
            ),
            (
                f"{weekly_series_per_week.min():,} to "
                f"{weekly_series_per_week.max():,}"
            ),
            f"{weekly_gap_count:,} weekly gaps",
            f"{weekly_target_mismatch_count:,} mismatches",
        ],
        "passed": [
            weekly_duplicate_key_count == 0,
            invalid_complete_week_count == 0,
            weekly_series_count == EXPECTED_SERIES_COUNT,
            weekly_records_per_series.eq(weekly_week_count).all(),
            weekly_series_per_week.eq(EXPECTED_SERIES_COUNT).all(),
            weekly_gap_count == 0,
            weekly_target_mismatch_count == 0,
        ],
    }
)

weekly_validation_register["status"] = np.where(
    weekly_validation_register["passed"],
    "PASS",
    "FAIL",
)

weekly_validation_register = weekly_validation_register[
    ["control", "expected", "observed", "status"]
]

display(weekly_validation_register)

failed_weekly_checks = weekly_validation_register.loc[
    weekly_validation_register["status"].eq("FAIL"),
    "control",
].tolist()

assert not failed_weekly_checks, (
    "Weekly-base validation failed for:\n- "
    + "\n- ".join(failed_weekly_checks)
)

print(
    "✅ Weekly modelling base validated: "
    f"{len(weekly_validation_register)} of "
    f"{len(weekly_validation_register)} controls passed."
)

,control,expected,observed,status
0,Unique SKU-store-week key,0 duplicates,0 duplicates,PASS
1,Complete seven-day weeks,0 invalid records,0 invalid records,PASS
2,Approved series retained,300 series,300 series,PASS
3,Records per series,273 for every series,273 to 273,PASS
4,Series per retail week,300 for every week,300 to 300,PASS
5,Continuous weekly chronology,0 weekly gaps,0 weekly gaps,PASS
6,Weekly target reconciliation,0 mismatches,0 mismatches,PASS


✅ Weekly modelling base validated: 7 of 7 controls passed.


### ✅ Weekly Modelling Base Result

The validated daily analytical base was successfully converted into **81,900 complete SKU-store-week observations** across **300 series and 273 consecutive retail weeks**. All records have a unique weekly key, complete seven-day coverage and continuous chronology. Weekly demand reconciles exactly with the corresponding daily demand, while 300 partial-week records were correctly excluded.

The weekly modelling base is approved for leakage-controlled feature engineering.

**✅ Section 4 is complete.**

-----

# 🧩 5. Forecasting Feature Engineering

This section engineers the model-ready predictors required for weekly demand forecasting. The features will represent calendar position, historical demand, commercial history, known calendar drivers, point-in-time demand behaviour and product-store hierarchy context.

All demand-, price- and promotion-derived features will use information from weeks preceding the target week. Target-week calendar, scheduled-event and applicable SNAP information may be used only because it is known before the forecast is created.

## 5.1 Calendar and Seasonal Features

Calendar features will represent chronological progression and recurring annual seasonality. These predictors are known in advance and do not depend on future demand outcomes.



In [20]:
# Create the working feature dataset
weekly_features = weekly_base.copy()

# Anchor calendar features to the start of each retail week
calendar_reference = weekly_features["week_start_date"]

# Represent chronological progression
weekly_features["time_index"] = (
    (calendar_reference - calendar_reference.min()).dt.days // 7
).astype("int16")

# Represent recurring annual seasonality
weekly_features["calendar_month"] = (
    calendar_reference.dt.month.astype("int8")
)

weekly_features["week_of_year"] = (
    calendar_reference.dt.isocalendar().week.astype("int8")
)

annual_angle = (
    2
    * np.pi
    * (weekly_features["week_of_year"] - 1)
    / 52.1775
)

weekly_features["week_of_year_sin"] = (
    np.sin(annual_angle).astype("float32")
)

weekly_features["week_of_year_cos"] = (
    np.cos(annual_angle).astype("float32")
)

CALENDAR_FEATURE_COLUMNS = [
    "time_index",
    "calendar_month",
    "week_of_year",
    "week_of_year_sin",
    "week_of_year_cos",
]

# Display one record per retail week
calendar_feature_preview = (
    weekly_features[
        [
            "wm_yr_wk",
            "week_start_date",
            *CALENDAR_FEATURE_COLUMNS,
        ]
    ]
    .drop_duplicates(subset=["wm_yr_wk"])
    .sort_values("week_start_date")
    .head(12)
)

calendar_feature_preview

,wm_yr_wk,week_start_date,time_index,calendar_month,week_of_year,week_of_year_sin,week_of_year_cos
0,11101,2011-01-29,0,1,4,0.35,0.94
1,11102,2011-02-05,1,2,5,0.46,0.89
2,11103,2011-02-12,2,2,6,0.57,0.82
3,11104,2011-02-19,3,2,7,0.66,0.75
4,11105,2011-02-26,4,2,8,0.75,0.67
5,11106,2011-03-05,5,3,9,0.82,0.57
6,11107,2011-03-12,6,3,10,0.88,0.47
7,11108,2011-03-19,7,3,11,0.93,0.36
8,11109,2011-03-26,8,3,12,0.97,0.24
9,11110,2011-04-02,9,4,13,0.99,0.13


### 📝 Calendar Feature Summary

Five calendar predictors were created successfully. The `time_index` advances sequentially across retail weeks, while calendar month and ISO week identify seasonal position. The sine and cosine features represent annual seasonality continuously across the year-end boundary.

Because all five features are derived solely from the retail-week start date, they are known before forecasting and introduce no target leakage.

## 5.2 Historical Demand Features

Demand lags and shifted historical statistics will summarize recent, medium-term and annual demand patterns. Each calculation will exclude the target week's demand so that only preceding observations contribute to its predictors.
> **Multi-step forecasting rule:** During an eight-week validation or future forecast, demand-derived features will be updated recursively using preceding observations or model predictions. Actual demand from inside the forecast period must never enter subsequent predictors.

In [21]:
# Ensure chronological ordering within every SKU-store series
weekly_features = (
    weekly_features
    .sort_values(["sku_id", "week_start_date"])
    .reset_index(drop=True)
)

demand_by_series = weekly_features.groupby(
    "sku_id",
    observed=True,
)["weekly_units_sold"]

# Create historical demand lags
for lag in DEMAND_LAGS:
    weekly_features[f"demand_lag_{lag}"] = (
        demand_by_series
        .shift(lag)
        .astype("float32")
    )

# Create shifted rolling-demand averages
for window in ROLLING_MEAN_WINDOWS:
    weekly_features[f"demand_rolling_mean_{window}"] = (
        demand_by_series
        .transform(
            lambda series, w=window: (
                series
                .shift(1)
                .rolling(window=w, min_periods=w)
                .mean()
            )
        )
        .astype("float32")
    )

# Create shifted rolling-demand volatility measures
for window in ROLLING_STD_WINDOWS:
    weekly_features[f"demand_rolling_std_{window}"] = (
        demand_by_series
        .transform(
            lambda series, w=window: (
                series
                .shift(1)
                .rolling(window=w, min_periods=w)
                .std()
            )
        )
        .astype("float32")
    )

HISTORICAL_DEMAND_FEATURE_COLUMNS = [
    *[f"demand_lag_{lag}" for lag in DEMAND_LAGS],
    *[
        f"demand_rolling_mean_{window}"
        for window in ROLLING_MEAN_WINDOWS
    ],
    *[
        f"demand_rolling_std_{window}"
        for window in ROLLING_STD_WINDOWS
    ],
]

# Preview records with complete annual-history features
example_sku = weekly_features["sku_id"].iloc[0]

historical_demand_preview = weekly_features.loc[
    weekly_features["sku_id"].eq(example_sku)
    & weekly_features["demand_lag_52"].notna(),
    [
        "sku_id",
        "wm_yr_wk",
        "week_start_date",
        "weekly_units_sold",
        "demand_lag_1",
        "demand_lag_4",
        "demand_lag_8",
        "demand_lag_13",
        "demand_lag_52",
        "demand_rolling_mean_4",
        "demand_rolling_mean_13",
        "demand_rolling_mean_52",
        "demand_rolling_std_13",
    ],
].head(8)

historical_demand_preview

,sku_id,wm_yr_wk,week_start_date,weekly_units_sold,demand_lag_1,demand_lag_4,demand_lag_8,demand_lag_13,demand_lag_52,demand_rolling_mean_4,demand_rolling_mean_13,demand_rolling_mean_52,demand_rolling_std_13
52,FOODS_1_001_CA_1,11201,2012-01-28,7,2.00,6.00,12.00,8.00,10.00,4.75,8.69,6.27,6.06
53,FOODS_1_001_CA_1,11202,2012-02-04,5,7.00,4.00,6.00,7.00,6.00,5.00,8.62,6.21,6.08
54,FOODS_1_001_CA_1,11203,2012-02-11,6,5.00,7.00,26.00,8.00,10.00,5.25,8.46,6.19,6.15
55,FOODS_1_001_CA_1,11204,2012-02-18,6,6.00,2.00,4.00,10.00,13.00,5.00,8.31,6.12,6.18
56,FOODS_1_001_CA_1,11205,2012-02-25,8,6.00,7.00,6.00,13.00,15.00,6.00,8.00,5.98,6.19
57,FOODS_1_001_CA_1,11206,2012-03-03,6,8.00,5.00,4.00,12.00,11.00,6.25,7.62,5.85,6.01
58,FOODS_1_001_CA_1,11207,2012-03-10,8,6.00,6.00,7.00,6.00,7.00,6.50,7.15,5.75,5.87
59,FOODS_1_001_CA_1,11208,2012-03-17,5,8.00,6.00,2.00,26.00,6.00,7.00,7.31,5.77,5.86


<div style="break-before: page; page-break-before: always;"></div>

### 📝 Historical Demand Feature Summary

Fifteen historical-demand predictors were created: seven demand lags, five shifted rolling means and three shifted rolling standard deviations. The preview begins at the first observation with 52 weeks of preceding history.

All rolling calculations exclude the target week through a one-week shift. Earlier missing values are therefore expected structural missingness and will be handled by the history-eligibility rule in Section 6.

## 5.3 Commercial and Known Calendar Drivers

Historical price and promotion features will use information from weeks preceding the target week. Scheduled-event and state-applicable SNAP features may describe the target week because their timing is known in advance.
> **Forecast-horizon rule:** Future price and promotion outcomes are not assumed to be known. Their historical features are therefore shifted by eight weeks, keeping them outside the complete forecast block. Target-week event and applicable SNAP information may be used directly because it is known in advance.

In [22]:
# Keep historical commercial information outside the 8-week forecast block
COMMERCIAL_HISTORY_SHIFT = 8

commercial_by_series = weekly_features.groupby(
    "sku_id",
    observed=True,
)

# Treat zero as unavailable and carry forward the last positive price
price_series_key = weekly_features["sku_id"]

last_observed_positive_price = (
    weekly_features["unit_price"]
    .where(weekly_features["unit_price"].gt(0))
    .groupby(price_series_key, observed=True)
    .ffill()
)

# Shift historical prices beyond the complete forecast block
price_lag_8_raw = (
    last_observed_positive_price
    .groupby(price_series_key, observed=True)
    .shift(COMMERCIAL_HISTORY_SHIFT)
)

price_lag_9_raw = (
    last_observed_positive_price
    .groupby(price_series_key, observed=True)
    .shift(COMMERCIAL_HISTORY_SHIFT + 1)
)

# Use zero only when no positive historical price has been observed
weekly_features["price_lag_8"] = (
    price_lag_8_raw
    .fillna(0.0)
    .astype("float32")
)

# Calculate change only when both historical prices are available
valid_price_comparison = (
    price_lag_8_raw.notna()
    & price_lag_9_raw.notna()
)

weekly_features["price_change_lag_8"] = (
    price_lag_8_raw
    .div(price_lag_9_raw)
    .sub(1)
    .where(valid_price_comparison, 0.0)
    .astype("float32")
)

# Average available historical prices over four weeks
weekly_features["price_rolling_mean_4_lag_8"] = (
    price_lag_8_raw
    .groupby(price_series_key, observed=True)
    .transform(
        lambda series: (
            series
            .rolling(
                window=4,
                min_periods=1,
            )
            .mean()
        )
    )
    .fillna(0.0)
    .astype("float32")
)

# Create historical promotion features
weekly_features["promo_lag_8"] = (
    commercial_by_series["promo_week_flag"]
    .shift(COMMERCIAL_HISTORY_SHIFT)
    .astype("float32")
)

weekly_features["promo_rolling_rate_4_lag_8"] = (
    commercial_by_series["promo_week_flag"]
    .transform(
        lambda series: (
            series
            .shift(COMMERCIAL_HISTORY_SHIFT)
            .rolling(
                window=4,
                min_periods=4,
            )
            .mean()
        )
    )
    .astype("float32")
)

# Define approved historical commercial features
HISTORICAL_COMMERCIAL_FEATURE_COLUMNS = [
    "price_lag_8",
    "price_change_lag_8",
    "price_rolling_mean_4_lag_8",
    "promo_lag_8",
    "promo_rolling_rate_4_lag_8",
]

# Retain only non-redundant target-week drivers known in advance
KNOWN_CALENDAR_DRIVER_COLUMNS = [
    "event_days",
    "cultural_event_flag",
    "national_event_flag",
    "religious_event_flag",
    "sporting_event_flag",
    "snap_active_days",
]

# Combine the commercial and known calendar feature groups
COMMERCIAL_AND_CALENDAR_FEATURE_COLUMNS = [
    *HISTORICAL_COMMERCIAL_FEATURE_COLUMNS,
    *KNOWN_CALENDAR_DRIVER_COLUMNS,
]

# Preview features with current-week fields included only as references
commercial_driver_preview = weekly_features.loc[
    weekly_features["sku_id"].eq(example_sku)
    & weekly_features["demand_lag_52"].notna(),
    [
        "sku_id",
        "wm_yr_wk",
        "week_start_date",
        "unit_price",
        "price_lag_8",
        "price_change_lag_8",
        "price_rolling_mean_4_lag_8",
        "promo_week_flag",
        "promo_lag_8",
        "promo_rolling_rate_4_lag_8",
        "event_days",
        "snap_active_days",
    ],
].head(8)

commercial_driver_preview

,sku_id,wm_yr_wk,week_start_date,unit_price,price_lag_8,price_change_lag_8,price_rolling_mean_4_lag_8,promo_week_flag,promo_lag_8,promo_rolling_rate_4_lag_8,event_days,snap_active_days
52,FOODS_1_001_CA_1,11201,2012-01-28,2.00,2.00,0.00,2.00,0,0.00,0.00,0,3
53,FOODS_1_001_CA_1,11202,2012-02-04,2.00,2.00,0.00,2.00,0,0.00,0.00,1,7
54,FOODS_1_001_CA_1,11203,2012-02-11,2.00,2.00,0.00,2.00,0,0.00,0.00,1,0
55,FOODS_1_001_CA_1,11204,2012-02-18,2.00,2.00,0.00,2.00,0,0.00,0.00,2,0
56,FOODS_1_001_CA_1,11205,2012-02-25,2.00,2.00,0.00,2.00,0,0.00,0.00,1,2
57,FOODS_1_001_CA_1,11206,2012-03-03,2.00,2.00,0.00,2.00,0,0.00,0.00,1,7
58,FOODS_1_001_CA_1,11207,2012-03-10,2.00,2.00,0.00,2.00,0,0.00,0.00,0,1
59,FOODS_1_001_CA_1,11208,2012-03-17,2.00,2.00,0.00,2.00,0,0.00,0.00,1,0


### 📝 Commercial and Calendar Driver Summary

Five historical commercial predictors and six known calendar drivers were prepared. Unavailable zero-price records are not treated as genuine selling prices; price features instead use the last positive price observed by the historical cutoff. Price change is recorded as zero when no valid prior comparison exists, while the lagged price remains zero when no positive price has yet been observed.

All price and promotion features remain at least eight weeks behind the target. Scheduled-event and applicable SNAP features describe the target week directly because their timing is known before forecasting.

## 5.4 Demand-Behaviour and Hierarchy Features

Point-in-time demand-behaviour features will describe prior selling activity, demand recency and intermittency without using full-series classifications. Stable product and store hierarchy fields will be retained as categorical context for the global forecasting models.

In [23]:
# Identify historically active demand weeks
active_demand_flag = (
    weekly_features["weekly_units_sold"]
    .gt(0)
    .astype("int8")
)

series_key = weekly_features["sku_id"]

# Count active weeks preceding each target week
prior_active_flag = (
    active_demand_flag
    .groupby(series_key, observed=True)
    .shift(1)
    .fillna(0)
    .astype("int8")
)

weekly_features["prior_active_week_count"] = (
    prior_active_flag
    .groupby(series_key, observed=True)
    .cumsum()
    .astype("int16")
)

# Measure demand recency using only preceding sales activity
active_week_index = weekly_features["time_index"].where(
    active_demand_flag.eq(1)
)

last_prior_active_index = (
    active_week_index
    .groupby(series_key, observed=True)
    .transform(lambda series: series.shift(1).ffill())
)

preceding_week_count = (
    weekly_features
    .groupby("sku_id", observed=True)
    .cumcount()
    .astype("int16")
)

weeks_since_last_sale = (
    weekly_features["time_index"]
    - last_prior_active_index
)

# Before the first sale, recency equals available preceding history
weekly_features["weeks_since_last_sale"] = (
    weeks_since_last_sale
    .fillna(preceding_week_count)
    .astype("int16")
)

# Represent recent demand intermittency
weekly_features["active_week_rate_13"] = (
    active_demand_flag
    .groupby(series_key, observed=True)
    .transform(
        lambda series: (
            series
            .shift(1)
            .rolling(window=13, min_periods=13)
            .mean()
        )
    )
    .astype("float32")
)

DEMAND_BEHAVIOUR_FEATURE_COLUMNS = [
    "prior_active_week_count",
    "weeks_since_last_sale",
    "active_week_rate_13",
]

# Retain hierarchy fields for later pipeline encoding
HIERARCHY_FEATURE_COLUMNS = [
    "item_id",
    "category",
    "department",
    "store_id",
    "state_id",
]

behaviour_hierarchy_preview = weekly_features.loc[
    weekly_features["sku_id"].eq(example_sku)
    & weekly_features["demand_lag_52"].notna(),
    [
        "sku_id",
        "wm_yr_wk",
        "week_start_date",
        "weekly_units_sold",
        *DEMAND_BEHAVIOUR_FEATURE_COLUMNS,
        *HIERARCHY_FEATURE_COLUMNS,
    ],
].head(8)

behaviour_hierarchy_preview

,sku_id,wm_yr_wk,week_start_date,weekly_units_sold,prior_active_week_count,weeks_since_last_sale,active_week_rate_13,item_id,category,department,store_id,state_id
52,FOODS_1_001_CA_1,11201,2012-01-28,7,40,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
53,FOODS_1_001_CA_1,11202,2012-02-04,5,41,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
54,FOODS_1_001_CA_1,11203,2012-02-11,6,42,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
55,FOODS_1_001_CA_1,11204,2012-02-18,6,43,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
56,FOODS_1_001_CA_1,11205,2012-02-25,8,44,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
57,FOODS_1_001_CA_1,11206,2012-03-03,6,45,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
58,FOODS_1_001_CA_1,11207,2012-03-10,8,46,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA
59,FOODS_1_001_CA_1,11208,2012-03-17,5,47,1,1.00,FOODS_1_001,FOODS,FOODS_1,CA_1,CA


### 📝 Demand-Behaviour and Hierarchy Feature Summary

Three point-in-time demand-behaviour predictors were created to represent prior selling activity, demand recency and recent intermittency. For the displayed observations, continuous recent sales cause the prior active-week count to increase, the weeks since the last sale to remain at one and the preceding 13-week activity rate to remain at 1.00.

Five stable product-store hierarchy fields were retained for encoding within the later modelling pipeline. All demand-behaviour calculations exclude the target week's outcome and therefore introduce no target leakage.

**✅ Section 5 is complete.**

------

# 🗂️ 6. Final Modelling Dataset

This section applies the minimum-history rule and organizes eligible weekly observations into the identifiers, target and approved numerical and categorical predictors required for modelling.

Missing values produced before sufficient lag or rolling history is available are structural and will not be replaced with zero.

## 6.1 Determine History Eligibility

The number of complete retail weeks preceding each observation will be calculated. Observations with at least 52 preceding weeks will be retained for the primary forecasting workflow, ensuring that every approved demand-history feature is available.


In [24]:
# Ensure chronological ordering within every SKU-store series
weekly_features = (
    weekly_features
    .sort_values(["sku_id", "week_start_date"])
    .reset_index(drop=True)
)

MIN_HISTORY_WEEKS = 52

# Count complete retail weeks preceding each observation
weekly_features["history_weeks_available"] = (
    weekly_features
    .groupby("sku_id", observed=True)
    .cumcount()
    .astype("int16")
)

# Identify observations with sufficient preceding history
weekly_features["history_eligible"] = (
    weekly_features["history_weeks_available"]
    .ge(MIN_HISTORY_WEEKS)
)

eligible_weekly_features = (
    weekly_features.loc[
        weekly_features["history_eligible"]
    ]
    .copy()
    .reset_index(drop=True)
)

eligible_records_per_series = (
    eligible_weekly_features
    .groupby("sku_id", observed=True)
    .size()
)

history_eligibility_summary = pd.DataFrame(
    {
        "metric": [
            "Complete weekly records assessed",
            "Structural-history records excluded",
            "History-eligible records retained",
            "History-eligible retail weeks",
            "SKU-store series retained",
            "Eligible records per series",
            "First eligible week start",
            "Last eligible week end",
        ],
        "value": [
            f"{len(weekly_features):,}",
            f"{(~weekly_features['history_eligible']).sum():,}",
            f"{len(eligible_weekly_features):,}",
            (
                f"{eligible_weekly_features['wm_yr_wk'].nunique():,}"
            ),
            (
                f"{eligible_weekly_features['sku_id'].nunique():,}"
            ),
            (
                f"{eligible_records_per_series.min():,} to "
                f"{eligible_records_per_series.max():,}"
            ),
            str(
                eligible_weekly_features[
                    "week_start_date"
                ].min().date()
            ),
            str(
                eligible_weekly_features[
                    "week_end_date"
                ].max().date()
            ),
        ],
    }
)

display(history_eligibility_summary)

# Show the eligibility boundary for one example series
history_eligibility_preview = weekly_features.loc[
    weekly_features["sku_id"].eq(example_sku)
    & weekly_features["history_weeks_available"].between(50, 54),
    [
        "sku_id",
        "wm_yr_wk",
        "week_start_date",
        "history_weeks_available",
        "history_eligible",
        "demand_lag_52",
        "demand_rolling_mean_52",
    ],
]

history_eligibility_preview

,metric,value
0,Complete weekly records assessed,"81,900"
1,Structural-history records excluded,"15,600"
2,History-eligible records retained,"66,300"
3,History-eligible retail weeks,221
4,SKU-store series retained,300
5,Eligible records per series,221 to 221
6,First eligible week start,2012-01-28
7,Last eligible week end,2016-04-22


,sku_id,wm_yr_wk,week_start_date,history_weeks_available,history_eligible,demand_lag_52,demand_rolling_mean_52
50,FOODS_1_001_CA_1,11151,2012-01-14,50,False,NaN,NaN
51,FOODS_1_001_CA_1,11152,2012-01-21,51,False,NaN,NaN
52,FOODS_1_001_CA_1,11201,2012-01-28,52,True,10.00,6.27
53,FOODS_1_001_CA_1,11202,2012-02-04,53,True,6.00,6.21
54,FOODS_1_001_CA_1,11203,2012-02-11,54,True,10.00,6.19


### 📝 History Eligibility Summary

The 52-week minimum-history rule excluded 15,600 structural-history observations and retained **66,300 eligible records** across all **300 SKU-store series**. Each series contributes 221 eligible weeks spanning 28 January 2012 through 22 April 2016.

The eligibility boundary behaves correctly: observations with 50 or 51 preceding weeks remain ineligible, while the observation with 52 preceding weeks is the first to contain complete annual-history features.


## 6.2 Finalize the Feature Schema

Eligible observations will be checked for remaining missing or non-finite predictor values. Identifiers and the weekly target will remain separate from the approved numerical and categorical feature lists used by the modelling pipeline.

In [25]:
# Define identifiers and target separately from model predictors
MODEL_IDENTIFIER_COLUMNS = [
    "sku_id",
    "wm_yr_wk",
    "week_start_date",
    "week_end_date",
]

TARGET_COLUMN = "weekly_units_sold"

# Combine the approved numerical feature groups
NUMERICAL_FEATURE_COLUMNS = [
    *CALENDAR_FEATURE_COLUMNS,
    *HISTORICAL_DEMAND_FEATURE_COLUMNS,
    *HISTORICAL_COMMERCIAL_FEATURE_COLUMNS,
    *KNOWN_CALENDAR_DRIVER_COLUMNS,
    *DEMAND_BEHAVIOUR_FEATURE_COLUMNS,
]

# Retain hierarchy fields for pipeline-based categorical encoding
CATEGORICAL_FEATURE_COLUMNS = [
    *HIERARCHY_FEATURE_COLUMNS,
]

MODEL_FEATURE_COLUMNS = [
    *NUMERICAL_FEATURE_COLUMNS,
    *CATEGORICAL_FEATURE_COLUMNS,
]

FINAL_MODELING_COLUMNS = [
    *MODEL_IDENTIFIER_COLUMNS,
    TARGET_COLUMN,
    *MODEL_FEATURE_COLUMNS,
]

# Confirm that the schema contains no duplicated columns
duplicated_schema_columns = sorted(
    {
        column
        for column in FINAL_MODELING_COLUMNS
        if FINAL_MODELING_COLUMNS.count(column) > 1
    }
)

assert not duplicated_schema_columns, (
    "Duplicated columns found in the modelling schema: "
    f"{duplicated_schema_columns}"
)

# Confirm that every required column exists
missing_schema_columns = sorted(
    set(FINAL_MODELING_COLUMNS)
    - set(eligible_weekly_features.columns)
)

assert not missing_schema_columns, (
    "Required modelling columns are missing: "
    f"{missing_schema_columns}"
)

# Confirm that identifiers and target are not included as predictors
forbidden_predictor_overlap = sorted(
    set(MODEL_FEATURE_COLUMNS)
    & set([*MODEL_IDENTIFIER_COLUMNS, TARGET_COLUMN])
)

assert not forbidden_predictor_overlap, (
    "Identifiers or target found among predictors: "
    f"{forbidden_predictor_overlap}"
)

# Construct the final in-memory modelling dataset
modeling_dataset = (
    eligible_weekly_features[
        FINAL_MODELING_COLUMNS
    ]
    .sort_values(["sku_id", "week_start_date"])
    .reset_index(drop=True)
    .copy()
)

# Check for unexpected missing or non-finite predictor values
predictor_missing_value_count = int(
    modeling_dataset[
        MODEL_FEATURE_COLUMNS
    ].isna().sum().sum()
)

numerical_feature_values = modeling_dataset[
    NUMERICAL_FEATURE_COLUMNS
].to_numpy(dtype="float64")

non_finite_numerical_value_count = int(
    (~np.isfinite(numerical_feature_values)).sum()
)

final_schema_summary = pd.DataFrame(
    {
        "metric": [
            "History-eligible records",
            "Identifier columns",
            "Target columns",
            "Numerical predictors",
            "Categorical predictors",
            "Total predictors",
            "Final modelling columns",
            "Missing predictor values",
            "Non-finite numerical values",
        ],
        "value": [
            f"{len(modeling_dataset):,}",
            f"{len(MODEL_IDENTIFIER_COLUMNS):,}",
            "1",
            f"{len(NUMERICAL_FEATURE_COLUMNS):,}",
            f"{len(CATEGORICAL_FEATURE_COLUMNS):,}",
            f"{len(MODEL_FEATURE_COLUMNS):,}",
            f"{len(FINAL_MODELING_COLUMNS):,}",
            f"{predictor_missing_value_count:,}",
            f"{non_finite_numerical_value_count:,}",
        ],
    }
)

display(final_schema_summary)

assert predictor_missing_value_count == 0, (
    "Unexpected missing predictor values remain."
)

assert non_finite_numerical_value_count == 0, (
    "Unexpected non-finite numerical values remain."
)

print("✅ Final modelling schema constructed successfully.")

,metric,value
0,History-eligible records,"66,300"
1,Identifier columns,4
2,Target columns,1
3,Numerical predictors,34
4,Categorical predictors,5
5,Total predictors,39
6,Final modelling columns,44
7,Missing predictor values,0
8,Non-finite numerical values,0


✅ Final modelling schema constructed successfully.


### 📝 Final Modelling Schema Summary

The final in-memory modelling dataset contains **66,300 history-eligible weekly observations** and **44 columns**: four identifiers, one target, 34 numerical predictors and five categorical predictors.

Schema checks confirmed that all required columns are present, identifiers and the target are excluded from the predictor set, and no duplicated schema columns exist. The 39 predictors contain no missing or non-finite values.

The dataset has not yet been exported; complete feature and leakage validation will be performed in Section 7 before export.

**✅ Section 6 is complete.**

-------

# ✅ 7. Feature Validation and Leakage Audit

This section validates the completed modelling dataset. Focused checks will verify structural integrity, feature-calculation accuracy and the absence of prohibited future information.

## 7.1 Structural Validation

The final dataset will be checked for unique SKU-week keys, chronological ordering within each series, complete retention of eligible series, valid non-negative targets, distinct feature names, complete identifiers and finite predictor values.


In [26]:
# Define the unique observation key
MODEL_KEY_COLUMNS = [
    "sku_id",
    "wm_yr_wk",
]

# Check for duplicated SKU-week observations
duplicate_key_count = int(
    modeling_dataset.duplicated(
        subset=MODEL_KEY_COLUMNS,
    ).sum()
)

# Check chronological ordering within every series
chronological_order_valid = bool(
    modeling_dataset
    .groupby("sku_id", observed=True)["week_start_date"]
    .apply(
        lambda series: (
            series.is_monotonic_increasing
            and series.is_unique
        )
    )
    .all()
)

# Confirm complete retention of eligible series
expected_eligible_series = set(
    eligible_weekly_features["sku_id"].unique()
)

retained_modeling_series = set(
    modeling_dataset["sku_id"].unique()
)

series_retention_valid = (
    retained_modeling_series
    == expected_eligible_series
)

# Confirm that every modelling record is history eligible
eligible_key_index = pd.MultiIndex.from_frame(
    eligible_weekly_features[
        MODEL_KEY_COLUMNS
    ].drop_duplicates()
)

modeling_key_index = pd.MultiIndex.from_frame(
    modeling_dataset[
        MODEL_KEY_COLUMNS
    ]
)

ineligible_record_count = int(
    (~modeling_key_index.isin(eligible_key_index)).sum()
)

# Check identifiers and targets
missing_identifier_value_count = int(
    modeling_dataset[
        MODEL_IDENTIFIER_COLUMNS
    ].isna().sum().sum()
)

target_values = modeling_dataset[
    TARGET_COLUMN
].to_numpy(dtype="float64")

missing_target_value_count = int(
    modeling_dataset[TARGET_COLUMN].isna().sum()
)

negative_target_value_count = int(
    (target_values < 0).sum()
)

# Check predictor structure and completeness
duplicated_feature_name_count = (
    len(MODEL_FEATURE_COLUMNS)
    - len(set(MODEL_FEATURE_COLUMNS))
)

missing_predictor_value_count = int(
    modeling_dataset[
        MODEL_FEATURE_COLUMNS
    ].isna().sum().sum()
)

numerical_validation_values = modeling_dataset[
    [TARGET_COLUMN, *NUMERICAL_FEATURE_COLUMNS]
].to_numpy(dtype="float64")

non_finite_numerical_value_count = int(
    (~np.isfinite(numerical_validation_values)).sum()
)

# Summarize the structural checks
structural_validation_summary = pd.DataFrame(
    {
        "check": [
            "Duplicated SKU-week keys",
            "Chronological ordering within series",
            "Eligible series retained",
            "Records outside the eligible set",
            "Missing identifier values",
            "Missing target values",
            "Negative target values",
            "Duplicated feature names",
            "Missing predictor values",
            "Non-finite numerical values",
        ],
        "result": [
            f"{duplicate_key_count:,}",
            (
                "Valid"
                if chronological_order_valid
                else "Invalid"
            ),
            (
                f"{len(retained_modeling_series):,} of "
                f"{len(expected_eligible_series):,}"
            ),
            f"{ineligible_record_count:,}",
            f"{missing_identifier_value_count:,}",
            f"{missing_target_value_count:,}",
            f"{negative_target_value_count:,}",
            f"{duplicated_feature_name_count:,}",
            f"{missing_predictor_value_count:,}",
            f"{non_finite_numerical_value_count:,}",
        ],
    }
)

display(structural_validation_summary)

# Enforce all structural requirements
assert duplicate_key_count == 0, (
    "Duplicated SKU-week keys were found."
)

assert chronological_order_valid, (
    "One or more series are not strictly chronological."
)

assert series_retention_valid, (
    "Eligible SKU-store series were lost or added."
)

assert ineligible_record_count == 0, (
    "Records outside the eligible set were found."
)

assert missing_identifier_value_count == 0, (
    "Missing identifier values were found."
)

assert missing_target_value_count == 0, (
    "Missing target values were found."
)

assert negative_target_value_count == 0, (
    "Negative target values were found."
)

assert duplicated_feature_name_count == 0, (
    "Duplicated feature names were found."
)

assert missing_predictor_value_count == 0, (
    "Missing predictor values were found."
)

assert non_finite_numerical_value_count == 0, (
    "Non-finite numerical values were found."
)

print("✅ Structural validation passed successfully.")

,check,result
0,Duplicated SKU-week keys,0
1,Chronological ordering within series,Valid
2,Eligible series retained,300 of 300
3,Records outside the eligible set,0
4,Missing identifier values,0
5,Missing target values,0
6,Negative target values,0
7,Duplicated feature names,0
8,Missing predictor values,0
9,Non-finite numerical values,0


✅ Structural validation passed successfully.


### 📝 Structural Validation Summary

The final modelling dataset contains unique SKU-week observations in strict chronological order and retains all **300 history-eligible SKU-store series**.

All records belong to the eligible observation set. Identifiers, targets and predictors are complete; targets are non-negative; feature names are distinct; and all numerical values are finite.

## 7.2 Feature Calculation Checks

Selected observations will be reconciled with their preceding historical source values to verify the alignment of demand lags, shifted rolling statistics, SNAP indicators and demand-recency features.

In [27]:
# Define representative features for source reconciliation
CALCULATION_CHECK_FEATURES = [
    "demand_lag_1",
    "demand_lag_52",
    "demand_rolling_mean_4",
    "demand_rolling_std_13",
    "snap_active_days",
    "weeks_since_last_sale",
]

missing_calculation_check_features = sorted(
    set(CALCULATION_CHECK_FEATURES)
    - set(MODEL_FEATURE_COLUMNS)
)

assert not missing_calculation_check_features, (
    "Required calculation-check features are missing: "
    f"{missing_calculation_check_features}"
)

# Recalculate selected demand-history features from weekly outcomes
source_series_key = weekly_features["sku_id"]
source_demand = weekly_features[TARGET_COLUMN]

expected_demand_lag_1 = (
    source_demand
    .groupby(source_series_key, observed=True)
    .shift(1)
)

expected_demand_lag_52 = (
    source_demand
    .groupby(source_series_key, observed=True)
    .shift(52)
)

expected_demand_rolling_mean_4 = (
    source_demand
    .groupby(source_series_key, observed=True)
    .transform(
        lambda series: (
            series
            .shift(1)
            .rolling(
                window=4,
                min_periods=4,
            )
            .mean()
        )
    )
)

expected_demand_rolling_std_13 = (
    source_demand
    .groupby(source_series_key, observed=True)
    .transform(
        lambda series: (
            series
            .shift(1)
            .rolling(
                window=13,
                min_periods=13,
            )
            .std()
        )
    )
)

# Recalculate applicable SNAP-active days from daily state indicators
daily_snap_source = analytics_daily[
    [
        "sku_id",
        "wm_yr_wk",
        "state_id",
        *STATE_SNAP_COLUMNS.values(),
    ]
].copy()

daily_snap_source["expected_snap_applicable"] = np.select(
    [
        daily_snap_source["state_id"].eq(state_id)
        for state_id in STATE_SNAP_COLUMNS
    ],
    [
        daily_snap_source[column]
        for column in STATE_SNAP_COLUMNS.values()
    ],
    default=np.nan,
)

assert daily_snap_source[
    "expected_snap_applicable"
].notna().all(), (
    "Some daily records could not be matched to a state SNAP field."
)

expected_snap_active_days = (
    daily_snap_source
    .groupby(
        MODEL_KEY_COLUMNS,
        observed=True,
    )["expected_snap_applicable"]
    .sum()
)

# Recalculate demand recency from preceding sales activity
source_active_demand = source_demand.gt(0)

source_active_week_index = weekly_features[
    "time_index"
].where(source_active_demand)

source_last_prior_active_index = (
    source_active_week_index
    .groupby(source_series_key, observed=True)
    .transform(
        lambda series: series.shift(1).ffill()
    )
)

source_preceding_week_count = (
    weekly_features
    .groupby("sku_id", observed=True)
    .cumcount()
)

expected_weeks_since_last_sale = (
    weekly_features["time_index"]
    - source_last_prior_active_index
).fillna(source_preceding_week_count)

# Assemble independently recalculated weekly values
expected_calculation_values = pd.DataFrame(
    {
        "demand_lag_1": expected_demand_lag_1,
        "demand_lag_52": expected_demand_lag_52,
        "demand_rolling_mean_4": (
            expected_demand_rolling_mean_4
        ),
        "demand_rolling_std_13": (
            expected_demand_rolling_std_13
        ),
        "weeks_since_last_sale": (
            expected_weeks_since_last_sale
        ),
    }
)

expected_calculation_values.index = (
    pd.MultiIndex.from_frame(
        weekly_features[MODEL_KEY_COLUMNS]
    )
)

expected_calculation_values["snap_active_days"] = (
    expected_snap_active_days
    .reindex(expected_calculation_values.index)
    .to_numpy()
)

expected_calculation_values = expected_calculation_values[
    CALCULATION_CHECK_FEATURES
]

# Align stored and independently recalculated values
observed_calculation_values = (
    modeling_dataset
    .set_index(MODEL_KEY_COLUMNS)[
        CALCULATION_CHECK_FEATURES
    ]
    .sort_index()
)

expected_calculation_values = (
    expected_calculation_values
    .reindex(observed_calculation_values.index)
)

assert expected_calculation_values.notna().all().all(), (
    "Expected source values are missing for eligible observations."
)

# Compare every eligible observation using numerical tolerance
calculation_check_labels = {
    "demand_lag_1": "One-week demand lag",
    "demand_lag_52": "Annual demand lag",
    "demand_rolling_mean_4": "Shifted 4-week demand mean",
    "demand_rolling_std_13": "Shifted 13-week demand deviation",
    "snap_active_days": "Applicable SNAP-active days",
    "weeks_since_last_sale": "Weeks since preceding sale",
}

calculation_check_records = []

for feature in CALCULATION_CHECK_FEATURES:
    observed_values = observed_calculation_values[
        feature
    ].to_numpy(dtype="float64")

    expected_values = expected_calculation_values[
        feature
    ].to_numpy(dtype="float64")

    absolute_differences = np.abs(
        observed_values - expected_values
    )

    matching_values = np.isclose(
        observed_values,
        expected_values,
        rtol=1e-5,
        atol=1e-6,
    )

    calculation_check_records.append(
        {
            "check": calculation_check_labels[feature],
            "records_checked": f"{len(observed_values):,}",
            "mismatched_records": (
                f"{(~matching_values).sum():,}"
            ),
            "maximum_absolute_difference": (
                f"{absolute_differences.max():.6f}"
            ),
        }
    )

feature_calculation_summary = pd.DataFrame(
    calculation_check_records
)

display(feature_calculation_summary)

# Show three representative observations for the example series
example_calculation_rows = (
    modeling_dataset.loc[
        modeling_dataset["sku_id"].eq(example_sku),
        [
            *MODEL_KEY_COLUMNS,
            "week_start_date",
        ],
    ]
    .reset_index(drop=True)
)

sample_positions = sorted(
    {
        0,
        len(example_calculation_rows) // 2,
        len(example_calculation_rows) - 1,
    }
)

calculation_preview_records = []

for _, sample_row in example_calculation_rows.iloc[
    sample_positions
].iterrows():
    sample_key = tuple(
        sample_row[column]
        for column in MODEL_KEY_COLUMNS
    )

    for feature in CALCULATION_CHECK_FEATURES:
        observed_value = observed_calculation_values.at[
            sample_key,
            feature,
        ]

        expected_value = expected_calculation_values.at[
            sample_key,
            feature,
        ]

        calculation_preview_records.append(
            {
                "sku_id": sample_row["sku_id"],
                "wm_yr_wk": sample_row["wm_yr_wk"],
                "week_start_date": (
                    sample_row["week_start_date"].date()
                ),
                "feature": feature,
                "stored_value": observed_value,
                "source_value": expected_value,
                "matches": bool(
                    np.isclose(
                        observed_value,
                        expected_value,
                        rtol=1e-5,
                        atol=1e-6,
                    )
                ),
            }
        )

feature_calculation_preview = pd.DataFrame(
    calculation_preview_records
)

display(feature_calculation_preview)

# Enforce calculation accuracy
total_calculation_mismatches = sum(
    int(record["mismatched_records"].replace(",", ""))
    for record in calculation_check_records
)

assert total_calculation_mismatches == 0, (
    "One or more selected feature calculations are misaligned."
)

print("✅ Feature calculation checks passed successfully.")

,check,records_checked,mismatched_records,maximum_absolute_difference
0,One-week demand lag,"66,300",0,0.000000
1,Annual demand lag,"66,300",0,0.000000
2,Shifted 4-week demand mean,"66,300",0,0.000000
3,Shifted 13-week demand deviation,"66,300",0,0.000004
4,Applicable SNAP-active days,"66,300",0,0.000000
5,Weeks since preceding sale,"66,300",0,0.000000


,sku_id,wm_yr_wk,week_start_date,feature,stored_value,source_value,matches
0,FOODS_1_001_CA_1,11201,2012-01-28,demand_lag_1,2.00,2.00,True
1,FOODS_1_001_CA_1,11201,2012-01-28,demand_lag_52,10.00,10.00,True
2,FOODS_1_001_CA_1,11201,2012-01-28,demand_rolling_mean_4,4.75,4.75,True
3,FOODS_1_001_CA_1,11201,2012-01-28,demand_rolling_std_13,6.06,6.06,True
4,FOODS_1_001_CA_1,11201,2012-01-28,snap_active_days,3.00,3.00,True
5,FOODS_1_001_CA_1,11201,2012-01-28,weeks_since_last_sale,1.00,1.00,True
6,FOODS_1_001_CA_1,11406,2014-03-08,demand_lag_1,10.00,10.00,True
7,FOODS_1_001_CA_1,11406,2014-03-08,demand_lag_52,5.00,5.00,True
8,FOODS_1_001_CA_1,11406,2014-03-08,demand_rolling_mean_4,6.50,6.50,True
9,FOODS_1_001_CA_1,11406,2014-03-08,demand_rolling_std_13,3.39,3.39,True


✅ Feature calculation checks passed successfully.


### 📝 Feature Calculation Check Summary

Six representative features were independently recalculated and reconciled across all **66,300 modelling observations**. The checks covered short- and annual-demand lags, shifted rolling statistics, applicable SNAP activity and demand recency.

No mismatches were found, and every maximum absolute difference was zero. Representative observations from the beginning, middle and end of an example series also matched their historical source values exactly.

## 7.3 Leakage Audit

The approved feature list and critical feature-generation rules will be audited to confirm that each predictor uses only preceding information or target-week information known independently in advance. Prohibited fields and unsafe alignments will be rejected through assertions.

In [28]:
# Register every predictor under an approved availability rule
FEATURE_AVAILABILITY_GROUPS = {
    "Calendar features": CALENDAR_FEATURE_COLUMNS,
    "Historical demand features": (
        HISTORICAL_DEMAND_FEATURE_COLUMNS
    ),
    "Historical commercial features": (
        HISTORICAL_COMMERCIAL_FEATURE_COLUMNS
    ),
    "Known calendar drivers": (
        KNOWN_CALENDAR_DRIVER_COLUMNS
    ),
    "Demand-behaviour features": (
        DEMAND_BEHAVIOUR_FEATURE_COLUMNS
    ),
    "Static hierarchy features": (
        CATEGORICAL_FEATURE_COLUMNS
    ),
}

feature_availability_rules = {
    "Calendar features": (
        "Target-week timing known in advance"
    ),
    "Historical demand features": (
        "Preceding demand observations only"
    ),
    "Historical commercial features": (
        "At least eight weeks behind the target"
    ),
    "Known calendar drivers": (
        "Target-week schedules known in advance"
    ),
    "Demand-behaviour features": (
        "Preceding demand activity only"
    ),
    "Static hierarchy features": (
        "Static product and store metadata"
    ),
}

feature_availability_summary = pd.DataFrame(
    {
        "feature_group": list(
            FEATURE_AVAILABILITY_GROUPS
        ),
        "availability_rule": [
            feature_availability_rules[group]
            for group in FEATURE_AVAILABILITY_GROUPS
        ],
        "feature_count": [
            len(columns)
            for columns in (
                FEATURE_AVAILABILITY_GROUPS.values()
            )
        ],
    }
)

display(feature_availability_summary)

# Check complete and exclusive feature-group registration
registered_feature_columns = [
    column
    for columns in FEATURE_AVAILABILITY_GROUPS.values()
    for column in columns
]

feature_membership_counts = pd.Series(
    registered_feature_columns
).value_counts()

overlapping_feature_count = int(
    feature_membership_counts.gt(1).sum()
)

unregistered_predictors = sorted(
    set(MODEL_FEATURE_COLUMNS)
    - set(registered_feature_columns)
)

unexpected_registered_features = sorted(
    set(registered_feature_columns)
    - set(MODEL_FEATURE_COLUMNS)
)

# Reject identifiers, outcomes and unknown target-week fields
PROHIBITED_PREDICTOR_COLUMNS = [
    *MODEL_IDENTIFIER_COLUMNS,
    TARGET_COLUMN,
    "unit_price",
    "promo_week_flag",
    "weekly_revenue",
]

prohibited_predictors_found = sorted(
    set(MODEL_FEATURE_COLUMNS)
    & set(PROHIBITED_PREDICTOR_COLUMNS)
)

unsafe_name_tokens = [
    "future_",
    "next_week",
    "_lead_",
]

unsafe_predictor_names = sorted(
    column
    for column in MODEL_FEATURE_COLUMNS
    if any(
        token in column.lower()
        for token in unsafe_name_tokens
    )
)

# Confirm that all explicitly named demand lags are historical
demand_lag_columns = [
    column
    for column in HISTORICAL_DEMAND_FEATURE_COLUMNS
    if column.startswith("demand_lag_")
]

invalid_demand_lag_names = [
    column
    for column in demand_lag_columns
    if not column.rsplit("_", 1)[-1].isdigit()
]

assert not invalid_demand_lag_names, (
    "Demand-lag periods could not be interpreted: "
    f"{invalid_demand_lag_names}"
)

demand_lag_periods = [
    int(column.rsplit("_", 1)[-1])
    for column in demand_lag_columns
]

minimum_demand_lag = min(demand_lag_periods)

# Confirm the approved target-week calendar drivers
APPROVED_KNOWN_CALENDAR_DRIVERS = [
    "event_days",
    "cultural_event_flag",
    "national_event_flag",
    "religious_event_flag",
    "sporting_event_flag",
    "snap_active_days",
]

known_calendar_policy_valid = (
    set(KNOWN_CALENDAR_DRIVER_COLUMNS)
    == set(APPROVED_KNOWN_CALENDAR_DRIVERS)
)

,feature_group,availability_rule,feature_count
0,Calendar features,Target-week timing known in advance,5
1,Historical demand features,Preceding demand observations only,15
2,Historical commercial features,At least eight weeks behind the target,5
3,Known calendar drivers,Target-week schedules known in advance,6
4,Demand-behaviour features,Preceding demand activity only,3
5,Static hierarchy features,Static product and store metadata,5


In [29]:
# Recalculate every historical commercial feature
commercial_audit_series_key = weekly_features["sku_id"]

audit_last_positive_price = (
    weekly_features["unit_price"]
    .where(weekly_features["unit_price"].gt(0))
    .groupby(
        commercial_audit_series_key,
        observed=True,
    )
    .ffill()
)

audit_price_lag_8_raw = (
    audit_last_positive_price
    .groupby(
        commercial_audit_series_key,
        observed=True,
    )
    .shift(COMMERCIAL_HISTORY_SHIFT)
)

audit_price_lag_9_raw = (
    audit_last_positive_price
    .groupby(
        commercial_audit_series_key,
        observed=True,
    )
    .shift(COMMERCIAL_HISTORY_SHIFT + 1)
)

audit_valid_price_comparison = (
    audit_price_lag_8_raw.notna()
    & audit_price_lag_9_raw.notna()
)

audit_expected_price_lag_8 = (
    audit_price_lag_8_raw.fillna(0.0)
)

audit_expected_price_change_lag_8 = (
    audit_price_lag_8_raw
    .div(audit_price_lag_9_raw)
    .sub(1)
    .where(audit_valid_price_comparison, 0.0)
)

audit_expected_price_mean = (
    audit_price_lag_8_raw
    .groupby(
        commercial_audit_series_key,
        observed=True,
    )
    .transform(
        lambda series: (
            series
            .rolling(
                window=4,
                min_periods=1,
            )
            .mean()
        )
    )
    .fillna(0.0)
)

audit_commercial_by_series = weekly_features.groupby(
    "sku_id",
    observed=True,
)

audit_expected_promo_lag_8 = (
    audit_commercial_by_series["promo_week_flag"]
    .shift(COMMERCIAL_HISTORY_SHIFT)
)

audit_expected_promo_rate = (
    audit_commercial_by_series["promo_week_flag"]
    .transform(
        lambda series: (
            series
            .shift(COMMERCIAL_HISTORY_SHIFT)
            .rolling(
                window=4,
                min_periods=4,
            )
            .mean()
        )
    )
)

expected_commercial_values = pd.DataFrame(
    {
        "price_lag_8": audit_expected_price_lag_8,
        "price_change_lag_8": (
            audit_expected_price_change_lag_8
        ),
        "price_rolling_mean_4_lag_8": (
            audit_expected_price_mean
        ),
        "promo_lag_8": audit_expected_promo_lag_8,
        "promo_rolling_rate_4_lag_8": (
            audit_expected_promo_rate
        ),
    }
)

expected_commercial_values.index = (
    pd.MultiIndex.from_frame(
        weekly_features[MODEL_KEY_COLUMNS]
    )
)

observed_commercial_values = (
    modeling_dataset
    .set_index(MODEL_KEY_COLUMNS)[
        HISTORICAL_COMMERCIAL_FEATURE_COLUMNS
    ]
    .sort_index()
)

expected_commercial_values = (
    expected_commercial_values
    .reindex(observed_commercial_values.index)[
        HISTORICAL_COMMERCIAL_FEATURE_COLUMNS
    ]
)

assert expected_commercial_values.notna().all().all(), (
    "Expected commercial-history values are missing."
)

# Compare stored commercial features with safe source values
commercial_audit_records = []

for feature in HISTORICAL_COMMERCIAL_FEATURE_COLUMNS:
    observed_values = observed_commercial_values[
        feature
    ].to_numpy(dtype="float64")

    expected_values = expected_commercial_values[
        feature
    ].to_numpy(dtype="float64")

    matching_values = np.isclose(
        observed_values,
        expected_values,
        rtol=1e-5,
        atol=1e-6,
    )

    commercial_audit_records.append(
        {
            "feature": feature,
            "records_checked": (
                f"{len(observed_values):,}"
            ),
            "mismatched_records": int(
                (~matching_values).sum()
            ),
        }
    )

commercial_leakage_summary = pd.DataFrame(
    commercial_audit_records
)

display(commercial_leakage_summary)

commercial_history_mismatch_count = int(
    commercial_leakage_summary[
        "mismatched_records"
    ].sum()
)

,feature,records_checked,mismatched_records
0,price_lag_8,"66,300",0
1,price_change_lag_8,"66,300",0
2,price_rolling_mean_4_lag_8,"66,300",0
3,promo_lag_8,"66,300",0
4,promo_rolling_rate_4_lag_8,"66,300",0


In [30]:
# Summarize the critical leakage restrictions
leakage_audit_summary = pd.DataFrame(
    {
        "check": [
            "Predictors audited",
            "Availability-registry coverage",
            "Feature-group overlaps",
            "Unregistered predictors",
            "Unexpected registered features",
            "Prohibited predictors",
            "Unsafe future or lead names",
            "Minimum demand lag",
            "Commercial-history shift",
            "Commercial-history mismatches",
            "Approved known calendar drivers",
            "Previous calculation mismatches",
            "Multi-step demand requirement",
        ],
        "result": [
            f"{len(MODEL_FEATURE_COLUMNS):,}",
            (
                f"{len(set(registered_feature_columns)):,} "
                f"of {len(MODEL_FEATURE_COLUMNS):,}"
            ),
            f"{overlapping_feature_count:,}",
            f"{len(unregistered_predictors):,}",
            f"{len(unexpected_registered_features):,}",
            f"{len(prohibited_predictors_found):,}",
            f"{len(unsafe_predictor_names):,}",
            f"{minimum_demand_lag} week",
            f"{COMMERCIAL_HISTORY_SHIFT} weeks",
            f"{commercial_history_mismatch_count:,}",
            (
                f"{len(KNOWN_CALENDAR_DRIVER_COLUMNS)} "
                f"of "
                f"{len(APPROVED_KNOWN_CALENDAR_DRIVERS)}"
            ),
            f"{total_calculation_mismatches:,}",
            "Recursive inference required",
        ],
    }
)

display(leakage_audit_summary)

# Enforce every leakage-control requirement
assert overlapping_feature_count == 0, (
    "One or more predictors belong to multiple groups."
)

assert not unregistered_predictors, (
    "Unregistered predictors were found: "
    f"{unregistered_predictors}"
)

assert not unexpected_registered_features, (
    "Unexpected registered features were found: "
    f"{unexpected_registered_features}"
)

assert not prohibited_predictors_found, (
    "Prohibited predictors were found: "
    f"{prohibited_predictors_found}"
)

assert not unsafe_predictor_names, (
    "Future- or lead-based predictor names were found: "
    f"{unsafe_predictor_names}"
)

assert minimum_demand_lag >= 1, (
    "A demand lag uses the current or a future outcome."
)

assert COMMERCIAL_HISTORY_SHIFT == 8, (
    "Commercial history does not remain outside "
    "the complete eight-week forecast block."
)

assert commercial_history_mismatch_count == 0, (
    "Historical commercial features are misaligned."
)

assert known_calendar_policy_valid, (
    "Unapproved target-week calendar drivers were found."
)

assert total_calculation_mismatches == 0, (
    "Previously checked historical calculations failed."
)

assert set(CATEGORICAL_FEATURE_COLUMNS) == set(
    HIERARCHY_FEATURE_COLUMNS
), (
    "Categorical predictors contain unapproved fields."
)

print("✅ Leakage audit passed successfully.")

,check,result
0,Predictors audited,39
1,Availability-registry coverage,39 of 39
2,Feature-group overlaps,0
3,Unregistered predictors,0
4,Unexpected registered features,0
5,Prohibited predictors,0
6,Unsafe future or lead names,0
7,Minimum demand lag,1 week
8,Commercial-history shift,8 weeks
9,Commercial-history mismatches,0


✅ Leakage audit passed successfully.


### 📝 Leakage Audit Summary

All **39 predictors** were assigned exactly once to one of six approved availability groups. No unregistered, overlapping, prohibited or future-named predictors were found.

Historical demand features use only preceding outcomes, while all five commercial-history features were independently reconciled across **66,300 observations** with zero mismatches and remain at least eight weeks behind the target. All six target-week calendar drivers are approved information known independently in advance.

All demand-derived predictors—including demand lags, rolling-demand statistics, prior active-week counts, demand recency and active-week rates—are safe for multi-step forecasting only when future notebooks recompute them recursively from information available at the forecast origin and from prior model predictions.

Precomputed feature values from inside a validation, test or future forecast block must never be used to predict its later horizons because those values may contain actual intermediate demand that was not available at the forecast origin.

**✅ Section 7 is complete. The validated modelling dataset is ready for controlled export.**

-----

# 💾 8. Export and Validate Modelling Outputs

This section exports the validated weekly modelling dataset and its supporting feature metadata. The persisted outputs will then be reloaded and verified as a reproducible handoff to the forecasting stage.

## 8.1 Export Modelling Outputs

The final modelling dataset, complete predictor list, categorical predictor list and feature dictionary will be exported to the dedicated `data/modeling` directory. The directory will be created only when the validated outputs are ready to be saved.


In [31]:
# Create the dedicated modelling-output directory
MODELING_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Define the final output paths
WEEKLY_FEATURES_FILE = (
    MODELING_DATA_DIR
    / "weekly_features_final.csv"
)

MODEL_FEATURE_LIST_FILE = (
    MODELING_DATA_DIR
    / "model_feature_list.json"
)

CATEGORICAL_FEATURES_FILE = (
    MODELING_DATA_DIR
    / "categorical_features.json"
)

FEATURE_DICTIONARY_FILE = (
    MODELING_DATA_DIR
    / "feature_dictionary.csv"
)

# Map each approved predictor to its feature group
feature_group_by_column = {
    column: feature_group
    for feature_group, columns in (
        FEATURE_AVAILABILITY_GROUPS.items()
    )
    for column in columns
}

availability_rule_by_column = {
    column: feature_availability_rules[feature_group]
    for feature_group, columns in (
        FEATURE_AVAILABILITY_GROUPS.items()
    )
    for column in columns
}

# Build metadata for every modelling-dataset column
feature_dictionary_records = []

for column_position, column in enumerate(
    modeling_dataset.columns,
    start=1,
):
    if column in MODEL_IDENTIFIER_COLUMNS:
        role = "Identifier"
        feature_group = "Observation structure"
        availability_rule = (
            "Observation identity or ordering; "
            "excluded from predictors"
        )

    elif column == TARGET_COLUMN:
        role = "Target"
        feature_group = "Forecasting outcome"
        availability_rule = (
            "Observed weekly outcome; "
            "excluded from predictors"
        )

    else:
        role = "Predictor"
        feature_group = feature_group_by_column[column]
        availability_rule = (
            availability_rule_by_column[column]
        )

    feature_dictionary_records.append(
        {
            "column_position": column_position,
            "column_name": column,
            "role": role,
            "feature_group": feature_group,
            "data_type": str(
                modeling_dataset[column].dtype
            ),
            "is_model_feature": (
                column in MODEL_FEATURE_COLUMNS
            ),
            "is_categorical_feature": (
                column in CATEGORICAL_FEATURE_COLUMNS
            ),
            "availability_rule": availability_rule,
        }
    )

feature_dictionary = pd.DataFrame(
    feature_dictionary_records
)

# Validate the prepared feature metadata
dictionary_predictors = feature_dictionary.loc[
    feature_dictionary["role"].eq("Predictor"),
    "column_name",
].tolist()

dictionary_categorical_features = (
    feature_dictionary.loc[
        feature_dictionary[
            "is_categorical_feature"
        ],
        "column_name",
    ].tolist()
)

assert len(feature_dictionary) == len(
    modeling_dataset.columns
), (
    "The feature dictionary does not cover "
    "every dataset column."
)

assert feature_dictionary[
    "column_name"
].tolist() == modeling_dataset.columns.tolist(), (
    "The feature-dictionary column order is incorrect."
)

assert dictionary_predictors == MODEL_FEATURE_COLUMNS, (
    "The dictionary predictor list does not match "
    "the approved model feature list."
)

assert (
    dictionary_categorical_features
    == CATEGORICAL_FEATURE_COLUMNS
), (
    "The dictionary categorical list does not match "
    "the approved categorical features."
)

assert feature_dictionary.notna().all().all(), (
    "Missing feature metadata was found."
)

print("✅ Export metadata prepared successfully.")
print(
    f"Dataset columns documented : "
    f"{len(feature_dictionary):,}"
)
print(
    f"Model predictors documented: "
    f"{len(dictionary_predictors):,}"
)

✅ Export metadata prepared successfully.
Dataset columns documented : 44
Model predictors documented: 39


In [32]:
# Export the validated weekly modelling dataset
modeling_dataset.to_csv(
    WEEKLY_FEATURES_FILE,
    index=False,
    date_format="%Y-%m-%d",
)

# Export the complete approved predictor list
with MODEL_FEATURE_LIST_FILE.open(
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        MODEL_FEATURE_COLUMNS,
        file,
        indent=2,
    )

# Export the approved categorical predictor list
with CATEGORICAL_FEATURES_FILE.open(
    mode="w",
    encoding="utf-8",
) as file:
    json.dump(
        CATEGORICAL_FEATURE_COLUMNS,
        file,
        indent=2,
    )

# Export the complete feature dictionary
feature_dictionary.to_csv(
    FEATURE_DICTIONARY_FILE,
    index=False,
)

# Register the exported outputs
exported_output_registry = {
    "Weekly modelling dataset": {
        "file_path": WEEKLY_FEATURES_FILE,
        "content": (
            f"{len(modeling_dataset):,} rows x "
            f"{modeling_dataset.shape[1]:,} columns"
        ),
    },
    "Model feature list": {
        "file_path": MODEL_FEATURE_LIST_FILE,
        "content": (
            f"{len(MODEL_FEATURE_COLUMNS):,} features"
        ),
    },
    "Categorical feature list": {
        "file_path": CATEGORICAL_FEATURES_FILE,
        "content": (
            f"{len(CATEGORICAL_FEATURE_COLUMNS):,} features"
        ),
    },
    "Feature dictionary": {
        "file_path": FEATURE_DICTIONARY_FILE,
        "content": (
            f"{len(feature_dictionary):,} rows x "
            f"{feature_dictionary.shape[1]:,} columns"
        ),
    },
}

# Confirm that every output was written successfully
export_summary_records = []

for output_name, details in (
    exported_output_registry.items()
):
    file_path = details["file_path"]
    file_exists = file_path.is_file()

    file_size_bytes = (
        file_path.stat().st_size
        if file_exists
        else 0
    )

    assert file_exists, (
        f"Exported file was not found: {file_path}"
    )

    assert file_size_bytes > 0, (
        f"Exported file is empty: {file_path}"
    )

    export_summary_records.append(
        {
            "output": output_name,
            "file_name": file_path.name,
            "content": details["content"],
            "file_size_kb": round(
                file_size_bytes / 1024,
                2,
            ),
            "status": "Exported",
        }
    )

export_summary = pd.DataFrame(
    export_summary_records
)

display(export_summary)

print("✅ Modelling outputs exported successfully.")
print(
    f"Output directory: "
    f"{MODELING_DATA_DIR.resolve()}"
)

,output,file_name,content,file_size_kb,status
0,Weekly modelling dataset,weekly_features_final.csv,"66,300 rows x 44 columns","16,833.70",Exported
1,Model feature list,model_feature_list.json,39 features,0.88,Exported
2,Categorical feature list,categorical_features.json,5 features,0.08,Exported
3,Feature dictionary,feature_dictionary.csv,44 rows x 8 columns,4.91,Exported


✅ Modelling outputs exported successfully.
Output directory: C:\Users\hp\Project FORESIGHT\data\modeling


### 📝 Modelling Output Export Summary

The validated modelling dataset was exported with **66,300 observations and 44 columns**, including **39 approved predictors** and **5 categorical predictors**.

The complete predictor list, categorical predictor list and **44-row feature dictionary** were also saved successfully. All four files are non-empty and available in the dedicated `data/modeling` directory for the forecasting-stage handoff.

## 8.2 Reload and Validate Outputs

All exported files will be reloaded to confirm their accessibility, dimensions, schemas, unique keys and consistency with the approved feature definitions. The final validation will also confirm that prohibited target-derived or future-information fields are absent from the model predictors.

In [33]:
# Reload the exported weekly modelling dataset
reloaded_modeling_dataset = pd.read_csv(
    WEEKLY_FEATURES_FILE,
    parse_dates=[
        "week_start_date",
        "week_end_date",
    ],
)
# Reload the approved model-feature list
with MODEL_FEATURE_LIST_FILE.open(
    mode="r",
    encoding="utf-8",
) as file:
    reloaded_model_feature_list = json.load(file)

# Reload the categorical-feature list
with CATEGORICAL_FEATURES_FILE.open(
    mode="r",
    encoding="utf-8",
) as file:
    reloaded_categorical_feature_list = json.load(file)

# Reload the feature dictionary
reloaded_feature_dictionary = pd.read_csv(
    FEATURE_DICTIONARY_FILE
)

# Summarize the reloaded outputs
reload_summary = pd.DataFrame(
    [
        {
            "output": "Weekly modelling dataset",
            "file_name": WEEKLY_FEATURES_FILE.name,
            "reloaded_content": (
                f"{len(reloaded_modeling_dataset):,} rows x "
                f"{reloaded_modeling_dataset.shape[1]:,} columns"
            ),
            "status": "Reloaded",
        },
        {
            "output": "Model feature list",
            "file_name": MODEL_FEATURE_LIST_FILE.name,
            "reloaded_content": (
                f"{len(reloaded_model_feature_list):,} features"
            ),
            "status": "Reloaded",
        },
        {
            "output": "Categorical feature list",
            "file_name": CATEGORICAL_FEATURES_FILE.name,
            "reloaded_content": (
                f"{len(reloaded_categorical_feature_list):,} features"
            ),
            "status": "Reloaded",
        },
        {
            "output": "Feature dictionary",
            "file_name": FEATURE_DICTIONARY_FILE.name,
            "reloaded_content": (
                f"{len(reloaded_feature_dictionary):,} rows x "
                f"{reloaded_feature_dictionary.shape[1]:,} columns"
            ),
            "status": "Reloaded",
        },
    ]
)

display(reload_summary)

assert reload_summary["status"].eq("Reloaded").all(), (
    "One or more modelling outputs could not be reloaded."
)

print("✅ All modelling outputs reloaded successfully.")

,output,file_name,reloaded_content,status
0,Weekly modelling dataset,weekly_features_final.csv,"66,300 rows x 44 columns",Reloaded
1,Model feature list,model_feature_list.json,39 features,Reloaded
2,Categorical feature list,categorical_features.json,5 features,Reloaded
3,Feature dictionary,feature_dictionary.csv,44 rows x 8 columns,Reloaded


✅ All modelling outputs reloaded successfully.


In [34]:
# Confirm dataset dimensions and column order
reloaded_dimensions_valid = (
    reloaded_modeling_dataset.shape
    == modeling_dataset.shape
)

reloaded_column_order_valid = (
    reloaded_modeling_dataset.columns.tolist()
    == modeling_dataset.columns.tolist()
)

assert reloaded_column_order_valid, (
    "The reloaded dataset column order has changed."
)

# Check unique observation keys
reloaded_duplicate_key_count = int(
    reloaded_modeling_dataset.duplicated(
        subset=MODEL_KEY_COLUMNS
    ).sum()
)

# Confirm chronological ordering within every series
reloaded_chronological_order_valid = bool(
    reloaded_modeling_dataset
    .groupby(
        "sku_id",
        sort=False,
        observed=True,
    )["week_start_date"]
    .apply(
        lambda series: (
            series.is_monotonic_increasing
            and series.is_unique
        )
    )
    .all()
)

# Confirm retention of the same SKU-store series
original_series = set(
    modeling_dataset["sku_id"].unique()
)

reloaded_series = set(
    reloaded_modeling_dataset["sku_id"].unique()
)

reloaded_series_retention_valid = (
    reloaded_series == original_series
)

# Check completeness and numerical validity
reloaded_missing_identifier_count = int(
    reloaded_modeling_dataset[
        MODEL_IDENTIFIER_COLUMNS
    ].isna().sum().sum()
)

reloaded_missing_target_count = int(
    reloaded_modeling_dataset[
        TARGET_COLUMN
    ].isna().sum()
)

reloaded_missing_predictor_count = int(
    reloaded_modeling_dataset[
        MODEL_FEATURE_COLUMNS
    ].isna().sum().sum()
)

reloaded_target_values = reloaded_modeling_dataset[
    TARGET_COLUMN
].to_numpy(dtype="float64")

reloaded_negative_target_count = int(
    (reloaded_target_values < 0).sum()
)

reloaded_numerical_values = reloaded_modeling_dataset[
    [
        TARGET_COLUMN,
        *NUMERICAL_FEATURE_COLUMNS,
    ]
].to_numpy(dtype="float64")

reloaded_non_finite_count = int(
    (~np.isfinite(reloaded_numerical_values)).sum()
)

# Confirm that values survived the CSV round trip
round_trip_values_match = True
round_trip_validation_error = ""

try:
    pd.testing.assert_frame_equal(
        modeling_dataset.reset_index(drop=True),
        reloaded_modeling_dataset.reset_index(drop=True),
        check_dtype=False,
        check_categorical=False,
        check_exact=False,
        check_freq=False,
        rtol=1e-5,
        atol=1e-6,
    )

except AssertionError as error:
    round_trip_values_match = False
    round_trip_validation_error = str(error)

# Summarize dataset-integrity checks
reloaded_dataset_validation_summary = pd.DataFrame(
    {
        "check": [
            "Dataset dimensions",
            "Column order",
            "Round-trip values",
            "Duplicated SKU-week keys",
            "Chronological ordering within series",
            "SKU-store series retained",
            "Missing identifier values",
            "Missing target values",
            "Missing predictor values",
            "Negative target values",
            "Non-finite numerical values",
        ],
        "result": [
            (
                f"{len(reloaded_modeling_dataset):,} rows x "
                f"{reloaded_modeling_dataset.shape[1]:,} columns"
            ),
            (
                "Valid"
                if reloaded_column_order_valid
                else "Invalid"
            ),
            (
                "Match"
                if round_trip_values_match
                else "Mismatch"
            ),
            f"{reloaded_duplicate_key_count:,}",
            (
                "Valid"
                if reloaded_chronological_order_valid
                else "Invalid"
            ),
            (
                f"{len(reloaded_series):,} of "
                f"{len(original_series):,}"
            ),
            f"{reloaded_missing_identifier_count:,}",
            f"{reloaded_missing_target_count:,}",
            f"{reloaded_missing_predictor_count:,}",
            f"{reloaded_negative_target_count:,}",
            f"{reloaded_non_finite_count:,}",
        ],
    }
)

display(reloaded_dataset_validation_summary)

# Enforce dataset-integrity requirements
assert reloaded_dimensions_valid, (
    "The reloaded dataset dimensions have changed."
)

assert round_trip_values_match, (
    "Reloaded values differ from the exported dataset. "
    f"{round_trip_validation_error}"
)

assert reloaded_duplicate_key_count == 0, (
    "Duplicated SKU-week keys were found after reloading."
)

assert reloaded_chronological_order_valid, (
    "Chronological ordering changed after reloading."
)

assert reloaded_series_retention_valid, (
    "SKU-store series were lost or added after reloading."
)

assert reloaded_missing_identifier_count == 0, (
    "Missing identifier values were found."
)

assert reloaded_missing_target_count == 0, (
    "Missing target values were found."
)

assert reloaded_missing_predictor_count == 0, (
    "Missing predictor values were found."
)

assert reloaded_negative_target_count == 0, (
    "Negative target values were found."
)

assert reloaded_non_finite_count == 0, (
    "Non-finite numerical values were found."
)

print("✅ Reloaded dataset integrity validated successfully.")

,check,result
0,Dataset dimensions,"66,300 rows x 44 columns"
1,Column order,Valid
2,Round-trip values,Match
3,Duplicated SKU-week keys,0
4,Chronological ordering within series,Valid
5,SKU-store series retained,300 of 300
6,Missing identifier values,0
7,Missing target values,0
8,Missing predictor values,0
9,Negative target values,0


✅ Reloaded dataset integrity validated successfully.


In [35]:
# Interpret the exported dictionary flags safely
reloaded_model_feature_flags = (
    reloaded_feature_dictionary[
        "is_model_feature"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

reloaded_categorical_feature_flags = (
    reloaded_feature_dictionary[
        "is_categorical_feature"
    ]
    .astype(str)
    .str.strip()
    .str.lower()
    .eq("true")
)

# Recover feature roles from the dictionary
reloaded_dictionary_identifiers = (
    reloaded_feature_dictionary.loc[
        reloaded_feature_dictionary[
            "role"
        ].eq("Identifier"),
        "column_name",
    ].tolist()
)

reloaded_dictionary_targets = (
    reloaded_feature_dictionary.loc[
        reloaded_feature_dictionary[
            "role"
        ].eq("Target"),
        "column_name",
    ].tolist()
)

reloaded_dictionary_predictors = (
    reloaded_feature_dictionary.loc[
        reloaded_feature_dictionary[
            "role"
        ].eq("Predictor"),
        "column_name",
    ].tolist()
)

reloaded_dictionary_model_features = (
    reloaded_feature_dictionary.loc[
        reloaded_model_feature_flags,
        "column_name",
    ].tolist()
)

reloaded_dictionary_categorical_features = (
    reloaded_feature_dictionary.loc[
        reloaded_categorical_feature_flags,
        "column_name",
    ].tolist()
)

# Check feature-list and dictionary agreement
model_feature_list_valid = (
    reloaded_model_feature_list
    == MODEL_FEATURE_COLUMNS
)

categorical_feature_list_valid = (
    reloaded_categorical_feature_list
    == CATEGORICAL_FEATURE_COLUMNS
)

dictionary_schema_valid = (
    reloaded_feature_dictionary.columns.tolist()
    == feature_dictionary.columns.tolist()
)

dictionary_column_coverage_valid = (
    reloaded_feature_dictionary[
        "column_name"
    ].tolist()
    == reloaded_modeling_dataset.columns.tolist()
)

dictionary_identifier_roles_valid = (
    reloaded_dictionary_identifiers
    == MODEL_IDENTIFIER_COLUMNS
)

dictionary_target_role_valid = (
    reloaded_dictionary_targets
    == [TARGET_COLUMN]
)

dictionary_predictor_roles_valid = (
    reloaded_dictionary_predictors
    == reloaded_model_feature_list
)

dictionary_model_flags_valid = (
    reloaded_dictionary_model_features
    == reloaded_model_feature_list
)

dictionary_categorical_flags_valid = (
    reloaded_dictionary_categorical_features
    == reloaded_categorical_feature_list
)

dictionary_missing_value_count = int(
    reloaded_feature_dictionary.isna().sum().sum()
)

# Confirm numerical predictors retained numerical schemas
reloaded_numerical_predictors = [
    column
    for column in reloaded_model_feature_list
    if column not in reloaded_categorical_feature_list
]

non_numeric_predictor_columns = [
    column
    for column in reloaded_numerical_predictors
    if not pd.api.types.is_numeric_dtype(
        reloaded_modeling_dataset[column]
    )
]

# Repeat the prohibited-information controls
reloaded_prohibited_predictors = sorted(
    set(reloaded_model_feature_list)
    & set(PROHIBITED_PREDICTOR_COLUMNS)
)

reloaded_unsafe_predictor_names = sorted(
    column
    for column in reloaded_model_feature_list
    if any(
        token in column.lower()
        for token in unsafe_name_tokens
    )
)

# Summarize metadata and handoff validation
reloaded_metadata_validation_summary = pd.DataFrame(
    {
        "check": [
            "Model feature-list agreement",
            "Categorical feature-list agreement",
            "Feature-dictionary rows",
            "Feature-dictionary schema",
            "Dataset-column dictionary coverage",
            "Identifier-role alignment",
            "Target-role alignment",
            "Predictor-role alignment",
            "Model-feature flag alignment",
            "Categorical-feature flag alignment",
            "Missing dictionary values",
            "Non-numeric numerical predictors",
            "Prohibited predictors",
            "Unsafe future or lead names",
            "Forecasting handoff",
        ],
        "result": [
            (
                f"{len(reloaded_model_feature_list):,} of "
                f"{len(MODEL_FEATURE_COLUMNS):,}"
            ),
            (
                f"{len(reloaded_categorical_feature_list):,} of "
                f"{len(CATEGORICAL_FEATURE_COLUMNS):,}"
            ),
            (
                f"{len(reloaded_feature_dictionary):,} of "
                f"{len(feature_dictionary):,}"
            ),
            (
                f"{reloaded_feature_dictionary.shape[1]:,} of "
                f"{feature_dictionary.shape[1]:,}"
            ),
            (
                f"{len(reloaded_feature_dictionary):,} of "
                f"{reloaded_modeling_dataset.shape[1]:,}"
            ),
            (
                "Valid"
                if dictionary_identifier_roles_valid
                else "Invalid"
            ),
            (
                "Valid"
                if dictionary_target_role_valid
                else "Invalid"
            ),
            (
                "Valid"
                if dictionary_predictor_roles_valid
                else "Invalid"
            ),
            (
                "Valid"
                if dictionary_model_flags_valid
                else "Invalid"
            ),
            (
                "Valid"
                if dictionary_categorical_flags_valid
                else "Invalid"
            ),
            f"{dictionary_missing_value_count:,}",
            f"{len(non_numeric_predictor_columns):,}",
            f"{len(reloaded_prohibited_predictors):,}",
            f"{len(reloaded_unsafe_predictor_names):,}",
            "Ready",
        ],
    }
)

display(reloaded_metadata_validation_summary)

# Enforce metadata and handoff requirements
assert model_feature_list_valid, (
    "The reloaded model feature list has changed."
)

assert categorical_feature_list_valid, (
    "The reloaded categorical feature list has changed."
)

assert dictionary_schema_valid, (
    "The reloaded feature-dictionary schema has changed."
)

assert dictionary_column_coverage_valid, (
    "The feature dictionary does not match "
    "the reloaded dataset columns."
)

assert dictionary_identifier_roles_valid, (
    "Identifier roles are misaligned."
)

assert dictionary_target_role_valid, (
    "The target role is misaligned."
)

assert dictionary_predictor_roles_valid, (
    "Predictor roles are misaligned."
)

assert dictionary_model_flags_valid, (
    "Model-feature flags are misaligned."
)

assert dictionary_categorical_flags_valid, (
    "Categorical-feature flags are misaligned."
)

assert dictionary_missing_value_count == 0, (
    "Missing feature metadata was found."
)

assert not non_numeric_predictor_columns, (
    "Numerical predictors were reloaded as non-numeric: "
    f"{non_numeric_predictor_columns}"
)

assert not reloaded_prohibited_predictors, (
    "Prohibited predictors were found: "
    f"{reloaded_prohibited_predictors}"
)

assert not reloaded_unsafe_predictor_names, (
    "Future- or lead-based predictor names were found: "
    f"{reloaded_unsafe_predictor_names}"
)

print("✅ Reloaded modelling outputs validated successfully.")
print("✅ Notebook 03 handoff is ready for forecasting.")

,check,result
0,Model feature-list agreement,39 of 39
1,Categorical feature-list agreement,5 of 5
2,Feature-dictionary rows,44 of 44
3,Feature-dictionary schema,8 of 8
4,Dataset-column dictionary coverage,44 of 44
5,Identifier-role alignment,Valid
6,Target-role alignment,Valid
7,Predictor-role alignment,Valid
8,Model-feature flag alignment,Valid
9,Categorical-feature flag alignment,Valid


✅ Reloaded modelling outputs validated successfully.
✅ Notebook 03 handoff is ready for forecasting.


In [36]:
# Create the reporting-table directory
REPORT_TABLES_DIR = (
    MODELING_DATA_DIR.parent.parent
    / "reports"
    / "tables"
)

REPORT_TABLES_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

# Define the validation-summary output path
FEATURE_VALIDATION_SUMMARY_FILE = (
    REPORT_TABLES_DIR
    / "03_feature_validation_summary.csv"
)

# Prepare the compact Notebook 03 validation register
feature_validation_records = [
    {
        "Validation Area": "Dataset Scope",
        "Metric": "Eligible modelling observations",
        "Result": f"{len(reloaded_modeling_dataset):,}",
        "Requirement": f"{len(modeling_dataset):,}",
        "Status": (
            "PASS"
            if len(reloaded_modeling_dataset)
            == len(modeling_dataset)
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Scope",
        "Metric": "Dataset columns",
        "Result": (
            f"{reloaded_modeling_dataset.shape[1]:,}"
        ),
        "Requirement": f"{modeling_dataset.shape[1]:,}",
        "Status": (
            "PASS"
            if reloaded_modeling_dataset.shape[1]
            == modeling_dataset.shape[1]
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Scope",
        "Metric": "SKU-store series",
        "Result": f"{len(reloaded_series):,}",
        "Requirement": f"{len(original_series):,}",
        "Status": (
            "PASS"
            if reloaded_series_retention_valid
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Registration",
        "Metric": "Approved model predictors",
        "Result": (
            f"{len(reloaded_model_feature_list):,}"
        ),
        "Requirement": f"{len(MODEL_FEATURE_COLUMNS):,}",
        "Status": (
            "PASS"
            if (
                model_feature_list_valid
                and dictionary_predictor_roles_valid
                and dictionary_model_flags_valid
            )
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Registration",
        "Metric": "Categorical predictors",
        "Result": (
            f"{len(reloaded_categorical_feature_list):,}"
        ),
        "Requirement": (
            f"{len(CATEGORICAL_FEATURE_COLUMNS):,}"
        ),
        "Status": (
            "PASS"
            if (
                categorical_feature_list_valid
                and dictionary_categorical_flags_valid
            )
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Registration",
        "Metric": "Feature-availability groups",
        "Result": (
            f"{len(FEATURE_AVAILABILITY_GROUPS):,}"
        ),
        "Requirement": "6",
        "Status": (
            "PASS"
            if len(FEATURE_AVAILABILITY_GROUPS) == 6
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Metadata",
        "Metric": "Feature-dictionary rows",
        "Result": (
            f"{len(reloaded_feature_dictionary):,}"
        ),
        "Requirement": (
            f"{reloaded_modeling_dataset.shape[1]:,}"
        ),
        "Status": (
            "PASS"
            if (
                dictionary_schema_valid
                and dictionary_column_coverage_valid
            )
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Column order",
        "Result": (
            "Valid"
            if reloaded_column_order_valid
            else "Invalid"
        ),
        "Requirement": "Valid",
        "Status": (
            "PASS"
            if reloaded_column_order_valid
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Duplicated SKU-week keys",
        "Result": f"{reloaded_duplicate_key_count:,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_duplicate_key_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Chronological ordering",
        "Result": (
            "Valid"
            if reloaded_chronological_order_valid
            else "Invalid"
        ),
        "Requirement": "Valid",
        "Status": (
            "PASS"
            if reloaded_chronological_order_valid
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Missing identifier values",
        "Result": (
            f"{reloaded_missing_identifier_count:,}"
        ),
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_missing_identifier_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Missing target values",
        "Result": f"{reloaded_missing_target_count:,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_missing_target_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Missing predictor values",
        "Result": (
            f"{reloaded_missing_predictor_count:,}"
        ),
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_missing_predictor_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Negative target values",
        "Result": f"{reloaded_negative_target_count:,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_negative_target_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Dataset Integrity",
        "Metric": "Non-finite numerical values",
        "Result": f"{reloaded_non_finite_count:,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if reloaded_non_finite_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Persistence",
        "Metric": "CSV round-trip values",
        "Result": (
            "Match"
            if round_trip_values_match
            else "Mismatch"
        ),
        "Requirement": "Match",
        "Status": (
            "PASS"
            if round_trip_values_match
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Metadata",
        "Metric": "Missing dictionary values",
        "Result": f"{dictionary_missing_value_count:,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if dictionary_missing_value_count == 0
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Feature Metadata",
        "Metric": "Non-numeric numerical predictors",
        "Result": f"{len(non_numeric_predictor_columns):,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if not non_numeric_predictor_columns
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Leakage Control",
        "Metric": "Prohibited predictors",
        "Result": f"{len(reloaded_prohibited_predictors):,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if not reloaded_prohibited_predictors
            else "FAIL"
        ),
    },
    {
        "Validation Area": "Leakage Control",
        "Metric": "Unsafe future or lead names",
        "Result": f"{len(reloaded_unsafe_predictor_names):,}",
        "Requirement": "0",
        "Status": (
            "PASS"
            if not reloaded_unsafe_predictor_names
            else "FAIL"
        ),
    },
]

# Determine the final forecasting-handoff status
forecasting_handoff_ready = all(
    record["Status"] == "PASS"
    for record in feature_validation_records
)

feature_validation_records.append(
    {
        "Validation Area": "Final Handoff",
        "Metric": "Forecasting handoff",
        "Result": (
            "Ready"
            if forecasting_handoff_ready
            else "Not Ready"
        ),
        "Requirement": "Ready",
        "Status": (
            "PASS"
            if forecasting_handoff_ready
            else "FAIL"
        ),
    }
)

feature_validation_summary = pd.DataFrame(
    feature_validation_records
)

# Export the validation summary
feature_validation_summary.to_csv(
    FEATURE_VALIDATION_SUMMARY_FILE,
    index=False,
)

# Reload and verify the exported report table
reloaded_feature_validation_summary = pd.read_csv(
    FEATURE_VALIDATION_SUMMARY_FILE,
    dtype=str,
)

pd.testing.assert_frame_equal(
    feature_validation_summary,
    reloaded_feature_validation_summary,
    check_dtype=False,
)

assert len(feature_validation_summary) == 21, (
    "The validation summary has an unexpected "
    "number of checks."
)

assert feature_validation_summary[
    "Status"
].eq("PASS").all(), (
    "One or more feature-validation checks failed."
)

assert FEATURE_VALIDATION_SUMMARY_FILE.is_file(), (
    "The feature-validation summary was not exported."
)

assert FEATURE_VALIDATION_SUMMARY_FILE.stat().st_size > 0, (
    "The exported feature-validation summary is empty."
)

display(feature_validation_summary)

print("✅ Feature-validation summary exported successfully.")
print(
    f"Validation checks: "
    f"{len(feature_validation_summary):,}"
)
print(
    f"Checks passed    : "
    f"{feature_validation_summary['Status'].eq('PASS').sum():,}"
)
print(
    f"Output file      : "
    f"{FEATURE_VALIDATION_SUMMARY_FILE.resolve()}"
)

,Validation Area,Metric,Result,Requirement,Status
0,Dataset Scope,Eligible modelling observations,"66,300","66,300",PASS
1,Dataset Scope,Dataset columns,44,44,PASS
2,Dataset Scope,SKU-store series,300,300,PASS
3,Feature Registration,Approved model predictors,39,39,PASS
4,Feature Registration,Categorical predictors,5,5,PASS
5,Feature Registration,Feature-availability groups,6,6,PASS
6,Feature Metadata,Feature-dictionary rows,44,44,PASS
7,Dataset Integrity,Column order,Valid,Valid,PASS
8,Dataset Integrity,Duplicated SKU-week keys,0,0,PASS
9,Dataset Integrity,Chronological ordering,Valid,Valid,PASS


✅ Feature-validation summary exported successfully.
Validation checks: 21
Checks passed    : 21
Output file      : C:\Users\hp\Project FORESIGHT\reports\tables\03_feature_validation_summary.csv


### 📝 Reload and Validation Summary

All four modelling outputs were reloaded successfully and matched their approved definitions.

The reloaded dataset retained all **66,300 observations, 44 columns and 300 SKU-store series**. Column order and round-trip values matched the original dataset, chronological ordering remained valid and no duplicated SKU-week keys were found.

All identifiers, target values and predictors were complete. No negative demand values, non-finite numerical values, prohibited predictors or unsafe future-information fields were detected.

The exported metadata also remained fully aligned, including all **39 model predictors**, **5 categorical predictors** and the complete **44-row feature dictionary**.

**✅ Section 8 is complete. Notebook 03 has produced a validated and reproducible modelling-data handoff ready for forecasting.**

------

# 🏁 9. Conclusion and Notebook 04 Handoff

Notebook 03 transformed the validated daily analytical data into a weekly, leakage-safe modelling dataset for Project FORESIGHT. The notebook completed weekly aggregation, target construction, feature engineering, history-eligibility filtering, predictor registration, leakage auditing and final output validation.

## 9.1 Work Completed and Key Outcomes

### Work Completed

Notebook 03 successfully:

- converted the validated daily analytical base into a consistent SKU-store-week modelling structure;
- created the weekly demand target;
- engineered calendar, cyclical, historical-demand, rolling, expanding, trend, momentum, commercial-history, demand-behaviour and hierarchy features;
- applied history-eligibility requirements before selecting modelling observations;
- documented every identifier, target and predictor in a complete feature dictionary;
- audited feature availability and historical alignment;
- exported and reloaded all modelling outputs;
- validated dimensions, ordering, keys, schemas, completeness and round-trip values.

### Final Modelling Dataset

The completed modelling dataset contains:

| Measure | Final Result |
|---|---:|
| Eligible modelling observations | **66,300** |
| Dataset columns | **44** |
| SKU-store series | **300** |
| Approved model predictors | **39** |
| Categorical predictors | **5** |
| Feature-availability groups | **6** |

All 300 SKU-store series were retained. The dataset contains no duplicated SKU-week keys, missing identifiers, missing target values, missing predictors, negative target values or non-finite numerical values.

## 9.2 Leakage Controls and Final Outputs

### Leakage-Control Results

All **39 predictors** were assigned exactly once to an approved availability group. No overlapping, unregistered, prohibited or unsafe future-named predictors were detected.

Historical-demand predictors use only preceding outcomes. All five commercial-history features were independently reconciled across **66,300 observations** with zero mismatches and remain at least eight weeks behind the target period. All six target-week calendar drivers are approved information known independently in advance.

All demand-derived predictors—including demand lags, rolling-demand statistics, prior active-week counts, demand recency and active-week rates—remain valid for multi-step forecasting only when future notebooks recompute them recursively using information available at the forecast origin and prior model predictions.

Precomputed feature values from inside a validation, test or future forecast block must never be used to predict its later horizons because those values may contain actual intermediate demand unavailable at the forecast origin.

### Validated Outputs

The following outputs were exported to `data/modeling` and successfully reloaded:

| Output File | Purpose |
|---|---|
| `weekly_features_final.csv` | Final weekly modelling dataset |
| `model_feature_list.json` | Ordered list of 39 approved predictors |
| `categorical_features.json` | Ordered list of 5 categorical predictors |
| `feature_dictionary.csv` | Metadata for all 44 dataset columns |

The exported files matched their approved dimensions, schemas, feature definitions and original in-memory values.

### Scope Boundary

Notebook 03 prepares the modelling data but does not train, compare or select forecasting models. Chronological evaluation, baseline forecasting and performance measurement belong to Notebook 04.



<div style="break-before: page; page-break-before: always;"></div>


## 9.3 Notebook 04 Handoff

Project FORESIGHT will now proceed to:

`04_Baseline_Forecasting.ipynb`

Notebook 04 will:

- reload and verify the validated modelling outputs;
- formalize the eight-week forecast horizon;
- establish chronological training, validation and test periods;
- define and apply WAPE and Forecast Bias as the locked primary evaluation metrics;
- implement naive, seasonal-naive, moving-average and exponential-smoothing baselines;
- perform rolling-origin validation;
- compare baseline performance and forecasting behaviour;
- select the champion baseline for later machine-learning comparison;
- export baseline forecasts, metrics and configuration metadata.

The chronological design must avoid random splitting and must preserve the leakage controls established in Notebook 03. At every rolling origin, all demand-derived predictors for later horizons must be rebuilt recursively using information available at that origin and prior model predictions. Actual intermediate demand from the validation, test or future forecast block must never be used.

WAPE and Forecast Bias must provide the primary evaluation view, while the seasonal-naive model must remain the principal benchmark for subsequent model evaluation.

**Final Status: Notebook 03 — Feature Engineering completed successfully.**

**✅ Section 9 is complete.**  
**✅ Notebook 03 has produced a validated, reproducible and leakage-safe modelling-data handoff ready for baseline forecasting.**

----